# NEXUS III v2.0.2 — Stage07A R4 final runnable notebook

Prospective protocol amendment dated 2026-09-30. Run A (discovery) first, then B (confirmation), each with **Runtime → Run all** on Colab A100. Use the same Google Drive account and scientific environment. Defaults execute the full research workload; there are no uploads or code edits required.

A freezes rank on 20 fresh tasks. B reads that lock and runs 30 disjoint tasks. Both use the separately identified R4 Drive folder. Do not import R2/R3 locks or task commits. Existing older results retain their original protocol.

R4 adds one stabilized fit-anchor subspace iteration, an unchanged-optimizer continuation, and signed baseline-relative causal gates. Original paired gates remain required. All seven future endpoint branches are saved and hash-verified. Cohorts, model, 500 updates/task, existing probe counts, surgery size, h=16 horizon, controls and original thresholds remain fixed. A prospective future-effect RMSE floor of 2e-5 is required in research mode.

The final cell prints the result ZIP path. If the runtime guard stops between tasks, rerun this same notebook to resume. Send the final exported ZIP(s) back for analysis. A failed scientific gate remains a valid outcome and does not justify threshold changes.

Validated with CPU engineering tests; A100/CUDA and full research duration remain untested. These notebooks test a task-local mechanism in the specified synthetic ResNet18-GN/AdamW system, not universal memory compression.


In [ ]:

# 0. Environment, deterministic execution, durable Google Drive persistence
import os, sys, json, math, time, random, hashlib, platform, copy, gc, zipfile, shutil, pickle, subprocess, warnings, itertools, marshal, types
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Tuple, Dict, List, Optional, Any
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import sklearn, scipy
from torch.func import functional_call, vjp, jvp
from scipy.stats import spearmanr
from sklearn.utils import resample

PROJECT="NEXUS-III"
VERSION="2.0.2"
IMPLEMENTATION_REVISION="STAGE07-R4-20260930-PROSPECTIVE-POWER-BASELINE"
PROFILE=os.environ.get("NEXUS_PROFILE","research").lower()
assert PROFILE in {"smoke","research"}
MASTER_SEED=20260928
STRICT_DETERMINISM=bool(int(os.environ.get("NEXUS_STRICT_DETERMINISM","1")))
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed:int):
    os.environ["PYTHONHASHSEED"]=str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(MASTER_SEED)
torch.use_deterministic_algorithms(True,warn_only=not STRICT_DETERMINISM)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
torch.set_float32_matmul_precision("highest")

REQUIRE_A100=bool(int(os.environ.get("NEXUS_REQUIRE_A100","1" if PROFILE=="research" else "0")))
if PROFILE=="research" and REQUIRE_A100:
    if not torch.cuda.is_available():
        raise RuntimeError("Stage07 research requires CUDA/A100; use smoke profile only for local QA.")
    gpu=torch.cuda.get_device_name(0)
    if "A100" not in gpu.upper():
        raise RuntimeError(f"Stage07 research requires A100 by default; detected {gpu}.")

_IN_COLAB=Path("/content").exists()
USE_DRIVE=bool(int(os.environ.get("NEXUS_USE_DRIVE","1" if (PROFILE=="research" and _IN_COLAB) else "0")))
ALLOW_LOCAL_RESEARCH=bool(int(os.environ.get("NEXUS_ALLOW_LOCAL_RESEARCH","0")))
if _IN_COLAB and USE_DRIVE and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive",force_remount=False)
    except Exception as e:
        if PROFILE=="research" and not ALLOW_LOCAL_RESEARCH:
            raise RuntimeError(f"Paid Stage07 research refused without durable Drive: {e}") from e

if Path("/content/drive/MyDrive").exists() and USE_DRIVE:
    ROOT=Path("/content/drive/MyDrive/NEXUS_III_v2_0_2_STAGE07_CONTROL_GEOMETRY_R4"); PERSISTENCE_ACTIVE="drive"
elif _IN_COLAB:
    if PROFILE=="research" and not ALLOW_LOCAL_RESEARCH:
        raise RuntimeError("Paid Stage07 research refused on ephemeral Colab storage.")
    ROOT=Path("/content/NEXUS_III_v2_0_2_STAGE07_CONTROL_GEOMETRY_R4"); PERSISTENCE_ACTIVE="local"
else:
    ROOT=Path("./NEXUS_III_v2_0_2_STAGE07_CONTROL_GEOMETRY_R4"); PERSISTENCE_ACTIVE="local"
ROOT.mkdir(parents=True,exist_ok=True)

if PROFILE=="research":
    try:
        import pyarrow as _pa
    except Exception:
        if _IN_COLAB:
            subprocess.check_call([sys.executable,"-m","pip","install","-q","pyarrow==23.0.1"])
            import pyarrow as _pa
        else:
            raise RuntimeError("Research mode requires pyarrow/Parquet.")
    PYARROW_VERSION=_pa.__version__
else:
    try:
        import pyarrow as _pa; PYARROW_VERSION=_pa.__version__
    except Exception:
        PYARROW_VERSION=None

try:
    _driver=subprocess.check_output(["nvidia-smi","--query-gpu=driver_version","--format=csv,noheader"],
                                    text=True,stderr=subprocess.DEVNULL).strip().splitlines()[0] if torch.cuda.is_available() else None
except Exception:
    _driver=None

ENV={
    "python":sys.version,"platform":platform.platform(),"torch":torch.__version__,"numpy":np.__version__,
    "pandas":pd.__version__,"sklearn":sklearn.__version__,"scipy":scipy.__version__,"pyarrow":PYARROW_VERSION,
    "device":str(DEVICE),"gpu_name":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_capability":list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
    "cuda":torch.version.cuda,"cudnn":torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    "nvidia_driver":_driver,"default_dtype":str(torch.get_default_dtype()),"deterministic":STRICT_DETERMINISM,
    "persistence":PERSISTENCE_ACTIVE,
}
# Cross-session compatibility intentionally excludes the host NVIDIA driver patch string.
# The exact driver remains recorded in ENV for provenance, while compatibility is defined by
# the scientific software stack, device family/capability, CUDA/cuDNN, dtype, and determinism.
ENV_COMPAT={k:ENV[k] for k in ("python","torch","numpy","pandas","sklearn","scipy","pyarrow","device","gpu_name","gpu_capability","cuda","cudnn","default_dtype","deterministic")}
ENV_COMPAT_HASH=hashlib.sha256(json.dumps(ENV_COMPAT,sort_keys=True).encode()).hexdigest()
print("DEVICE",DEVICE,ENV.get("gpu_name"))
print("PERSISTENCE",PERSISTENCE_ACTIVE)


In [ ]:

# 1. Frozen NEXUS-III Stage07 scientific protocol
@dataclass(frozen=True)
class CoreConfig:
    # Exact controlled system inherited from Stage06 unless explicitly stated here.
    train_steps:int=72 if PROFILE=="smoke" else 500
    batch_size:int=64 if PROFILE=="smoke" else 128
    lr:float=1.2e-3
    weight_decay:float=1e-4
    adam_beta1:float=0.9
    adam_beta2:float=0.999
    adam_eps:float=1e-8
    adam_amsgrad:bool=False
    adam_maximize:bool=False
    adam_foreach:bool=False
    adam_capturable:bool=False
    adam_differentiable:bool=False
    adam_fused:bool=False

    # Main control-geometry checkpoint and temporal diagnostic checkpoints.
    recipient_frac:float=0.70
    donor_lag_frac:float=0.30
    temporal_fracs:Tuple[float,...]=(0.30,0.50,0.70,0.90)
    future_horizons:Tuple[int,...]=(1,4,8) if PROFILE=="smoke" else (1,4,16)
    primary_future_horizon:int=8 if PROFILE=="smoke" else 16

    # Local causal-control operator J: delta m1 -> centered calibration logits after one common AdamW step.
    control_anchor_n:int=48 if PROFILE=="smoke" else 128
    basis_probes:int=8 if PROFILE=="smoke" else 24
    basis_power_iterations:int=1
    control_basis_tolerance:float=1e-6
    trace_probes:int=8 if PROFILE=="smoke" else 32
    transfer_trace_probes:int=8 if PROFILE=="smoke" else 32
    max_control_rank:int=6 if PROFILE=="smoke" else 16
    rank_candidates:Tuple[int,...]=(1,2,4,6) if PROFILE=="smoke" else (1,2,4,8,12,16)
    rank_discovery_mean_capture:float=0.80
    rank_discovery_ci_lower:float=0.70
    confirmation_capture_mean_min:float=0.80
    confirmation_capture_ci_lower_min:float=0.70
    confirmation_capture_task_fraction_min:float=0.80
    confirmation_capture_per_task_min:float=0.60
    low_rank_max_k:int=8

    # History privilege against strong matched controls.
    tensor_random_controls:int=3 if PROFILE=="smoke" else 8
    alignment_history_mean_min:float=0.15
    alignment_excess_ci_lower_min:float=0.05
    potency_random_ratio_ci_lower_min:float=5.0
    potency_random_ratio_mean_min:float=10.0
    potency_best_random_ratio_ci_lower_min:float=3.0
    potency_best_random_ratio_mean_min:float=5.0
    shuffled_ratio_ci_lower_min:float=1.5
    permuted_ratio_ci_lower_min:float=1.5

    # Finite causal decomposition. One alpha is calibrated from full history to a fixed relative m1 displacement;
    # the exact same alpha is used for P_U h and (I-P_U)h, so "capture" cannot be rescued by re-scaling pieces.
    causal_relative_m1_displacement:float=0.002
    causal_parallel_retention_ci_lower_min:float=0.60
    causal_parallel_retention_mean_min:float=0.70
    causal_parallel_cosine_ci_lower_min:float=0.75
    causal_perp_ratio_ci_upper_max:float=0.55
    causal_perp_ratio_mean_max:float=0.40
    causal_additivity_error_mean_max:float=0.35

    # Fresh-system competency (inherited from Stage06 population qualification).
    clean_median_acc_floor:float=0.25 if PROFILE=="smoke" else 0.65
    recipient_median_acc_floor:float=0.18 if PROFILE=="smoke" else 0.50
    clean_task_nontrivial_floor:float=0.20 if PROFILE=="smoke" else 0.40
    recipient_task_nontrivial_floor:float=0.15 if PROFILE=="smoke" else 0.35
    min_nontrivial_task_fraction:float=0.50 if PROFILE=="smoke" else 0.90

    # Per-task finite-surgery locality/effect qualification for H7-C.
    hidden_locality_min_secant_cosine:float=0.95 if PROFILE=="smoke" else 0.98
    hidden_locality_max_relative_norm_drift:float=0.35 if PROFILE=="smoke" else 0.25
    hidden_locality_required_fraction:float=1.0
    causal_min_one_step_rmse:float=1e-6 if PROFILE=="smoke" else 2e-5
    causal_min_future_rmse:float=1e-6 if PROFILE=="smoke" else 2e-5

    # Numerical/locality qualification.
    min_axis_norm:float=1e-20
    adjoint_rel_error_max:float=2e-4
    manual_adamw_param_max_abs_diff:float=2e-6
    finite_diff_rel_error_max:float=0.08
    bootstrap_n:int=300 if PROFILE=="smoke" else 10000
    signflip_mc_draws:int=20000 if PROFILE=="smoke" else 100000
    runtime_soft_limit_hours:float=11.25

CORE=CoreConfig()
PRIMARY_MODEL_SCALE=1 if PROFILE=="smoke" else 2

DISCOVERY_IDS=tuple(range(4000,4003)) if PROFILE=="smoke" else tuple(range(4000,4020))
CONFIRMATION_IDS=tuple(range(4100,4103)) if PROFILE=="smoke" else tuple(range(4100,4130))
assert set(DISCOVERY_IDS).isdisjoint(CONFIRMATION_IDS)

PARENT_EVIDENCE={
    "policy":"Design history only. No Stage01-06 response row, fitted coordinate, decoder, subspace, or confirmation outcome enters Stage07 estimation.",
    "NEXUS_I":"Stage1A negative; Stage1B d=1 geometry without predictive state; Stage1C promising pressure pilot; Stage2P/Q support preflights passed; architecture confirmation was attempted and stopped at ViT competency (optimistic median upper 0.612 < 0.65).",
    "NEXUS_II_RC11":"Pressure-induced emergence failed; clean eta=0 exhibited stable low-dimensional predictive structure and approximate closure.",
    "NEXUS_II_STAGE05":"RC12D H5 failed: F/I/Q/N/P/S passed while G/Z/E/C failed.",
    "NEXUS_II_STAGE06":{
        "protocol_hash":"89d065fb5d81d316667742c587edd1cdcf0b69c80a1e3a0a5adf9e4dafe434e1",
        "final_zip_sha256_reported":"a017f8ed47b68bbff089d2b4bb823f469508340cbff3fd35f1667d0642ba2729",
        "frozen_result":"H6-B=false, H6-W=false, H6-A=true. Within-task matched-query r=0.7765 [0.700,0.845], but causal future prediction lost to zero. Historical m1 potency was 90.93x median random and 70.79x best-of-eight random."
    }
}

PREREG={
    "scientific_question":"Does clean neural learning organize hidden AdamW first-moment memory into a genuinely low-rank, high-gain causal-control geometry that is distinct from predictive low-dimensional summaries?",
    "operator":"J_t maps an infinitesimal perturbation of AdamW exp_avg at fixed model parameters to centered logits after one common next AdamW update. The right control subspace is estimated on a calibration anchor without using historical m1, with one normalized and reorthogonalized J^T J subspace iteration and relative basis tolerance 1e-6; rank capture is selected and confirmed on an independent evaluation anchor under the same physical update.",
    "headline_claims":{
        "H7_F":"Fresh-system foundation: the frozen clean-learning population remains competent and every confirmation finite history surgery is local and function-preserving, with nontrivial one-step and h=16 full effects. The paired and both baseline-relative signed full effects must each have RMSE at least 2e-5 in research mode.",
        "H7_L":"Transferred low-rank control: a discovery-frozen rank k<=8 captures most Frobenius control energy on an independent output anchor, not merely the anchor used to estimate the subspace.",
        "H7_A":"History alignment/privilege: the historical recipient-minus-donor exp_avg direction aligns with and is more potent in the high-gain subspace than tensor-norm-matched random, sign-shuffled, and within-tensor-permuted controls. Current-gradient and preconditioned-gradient directions are frozen mechanistic comparators reported separately.",
        "H7_C":"Finite causal sufficiency: at the identical m1 surgery scale, P_U h preserves most h=16 functional effect while (I-P_U)h contributes little. The original paired-response gates AND the same retention/cosine/remainder/additivity gates for EACH signed effect relative to an unchanged-optimizer continuation must all pass. All seven endpoint branches use identical snapshots, batches and RNG. No even-response component is assumed negligible.",
    },
    "strong_claim":"H7_M = H7_F AND H7_L AND H7_A AND H7_C. Temporal measurements and gradient/preconditioned-gradient comparators are reported separately and cannot rescue H7_M.",
    "rank_selection":"On discovery only, choose the smallest candidate k whose mean independent-anchor control-energy capture is >=0.80 and bootstrap 95% lower bound >=0.70. If none qualifies, freeze the largest candidate and mark discovery low-rank prequalification false; confirmation still runs unchanged.",
    "confirmation_firewall":"Confirmation notebook may read only the cryptographically frozen discovery lock (selected rank, protocol/code/environment identities and preregistered selection diagnostics), never discovery raw response vectors when fitting confirmation quantities.",
    "cohort_independence":"Discovery locks task-identity hashes only (not outcomes). Confirmation must have unique task hashes and zero overlap with frozen discovery task hashes before claims are evaluated.",
    "no_rescue":"No post-confirmation threshold, rank, control panel, cohort, timepoint, or endpoint changes are permitted.",
}
PROSPECTIVE_AMENDMENT={"revision":"R4","date":"2026-09-30","parent_research_protocol_hash":"cc3ff07bd5b44534e99acba4fc5ac653062f13b5516031a6cb972d3ac2b979f2","reason":"Pre-outcome engineering calibration exposed weak-tail estimator sensitivity and cancellation of even finite responses. Freeze one subspace iteration, unchanged continuation, per-sign gates and endpoint evidence before new research.","no_reuse":"Do not import R3 commits/locks; any already collected R3 results remain a separate study.","scope":"Task-local recent-history control at the frozen checkpoint, displacement and h=16 in clean synthetic ResNet18-GN/AdamW; no universal basis, full optimizer-state compression or architecture/natural-data generalization claim."}
PROTOCOL_PAYLOAD={"amendment":PROSPECTIVE_AMENDMENT,"project":PROJECT,"version":VERSION,"core":asdict(CORE),"discovery_ids":list(DISCOVERY_IDS),"confirmation_ids":list(CONFIRMATION_IDS),"prereg":PREREG,"parent_evidence":PARENT_EVIDENCE}
PROTOCOL_HASH=hashlib.sha256(json.dumps(PROTOCOL_PAYLOAD,sort_keys=True,separators=(",",":")).encode()).hexdigest()
CONFIG_HASH=hashlib.sha256(json.dumps(asdict(CORE),sort_keys=True).encode()).hexdigest()
print("STAGE07 PROTOCOL HASH",PROTOCOL_HASH)


In [ ]:

# 2. Stable identity, hashing, atomic I/O, resumability, and <12h safety guard
def stable_seed(*parts)->int:
    return int(hashlib.sha256("|".join(map(str,parts)).encode()).hexdigest()[:8],16)

def tree_to_cpu(x):
    if torch.is_tensor(x): return x.detach().cpu().clone()
    if isinstance(x,dict): return {k:tree_to_cpu(v) for k,v in x.items()}
    if isinstance(x,list): return [tree_to_cpu(v) for v in x]
    if isinstance(x,tuple): return tuple(tree_to_cpu(v) for v in x)
    return copy.deepcopy(x)

def hash_tree(obj)->str:
    h=hashlib.sha256()
    def rec(x):
        if torch.is_tensor(x):
            y=x.detach().cpu().contiguous()
            h.update(b"T"); h.update(str(y.dtype).encode()); h.update(str(tuple(y.shape)).encode()); h.update(y.numpy().tobytes())
        elif isinstance(x,dict):
            h.update(b"D")
            for k in sorted(x,key=lambda z:repr(z)): rec(k); rec(x[k])
        elif isinstance(x,(list,tuple)):
            h.update(type(x).__name__.encode())
            for v in x: rec(v)
        elif x is None: h.update(b"N")
        else: h.update(type(x).__name__.encode()); h.update(repr(x).encode())
    rec(obj); return h.hexdigest()

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

def atomic_json(path,payload):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=Path(str(path)+".tmp")
    with open(tmp,"w") as f:
        json.dump(payload,f,indent=2,sort_keys=True); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)
    if PERSISTENCE_ACTIVE=="drive":
        try: os.sync()
        except Exception: pass

def save_table(df,stem):
    stem=Path(stem); stem.parent.mkdir(parents=True,exist_ok=True)
    pq=Path(str(stem)+".parquet"); tmp=Path(str(pq)+".tmp")
    if PROFILE=="research":
        df.to_parquet(tmp,index=False,compression="zstd"); os.replace(tmp,pq); return pq
    try:
        df.to_parquet(tmp,index=False,compression="zstd"); os.replace(tmp,pq); return pq
    except Exception:
        if tmp.exists(): tmp.unlink()
        cv=Path(str(stem)+".csv"); tmpc=Path(str(cv)+".tmp"); df.to_csv(tmpc,index=False); os.replace(tmpc,cv); return cv

def load_table(stem):
    stem=Path(stem)
    pq,cv=Path(str(stem)+".parquet"),Path(str(stem)+".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(stem)

RUN_STARTED_MONO=time.monotonic()
def runtime_hours(): return (time.monotonic()-RUN_STARTED_MONO)/3600.0

def _completed_task_seconds():
    vals=[]
    for cohort in ("discovery","confirmation"):
        d=BASE/cohort
        if not d.exists(): continue
        for p in d.glob("rep_*_commit.json"):
            try:
                x=json.loads(p.read_text()); s=float(x.get("seconds",float("nan")))
                if np.isfinite(s) and s>0: vals.append(s)
            except Exception: pass
    return vals

def guard_runtime_before_new_paid_task():
    if PROFILE!="research": return
    elapsed=runtime_hours()
    past=_completed_task_seconds()
    # Conservative task reserve: at least 30 minutes; once observed, 1.75x the
    # slowest completed task (capped only below by 30 min, never by an optimistic upper bound).
    reserve_h=max(0.50,(1.75*max(past)/3600.0) if past else 0.75)
    if elapsed>=CORE.runtime_soft_limit_hours or elapsed+reserve_h>=11.90:
        raise RuntimeError(
            f"Stage07 runtime guard refused a new paid task at {elapsed:.2f}h "
            f"(conservative task reserve {reserve_h:.2f}h). Completed tasks are durable; "
            "rerun the same notebook to resume. Scientific workload is NOT reduced."
        )

def _finite_vector(vals,label="measurement"):
    x=np.asarray(list(vals),dtype=float)
    if x.ndim!=1 or len(x)==0 or not np.isfinite(x).all():
        raise RuntimeError(f"{label}: expected a nonempty finite vector; refusing silent task exclusion.")
    return x

def _validated_capture(numerator,trace_estimate):
    numerator=float(numerator); trace_estimate=float(trace_estimate)
    if not np.isfinite(numerator) or not np.isfinite(trace_estimate) or numerator<0 or trace_estimate<=0:
        raise RuntimeError("Invalid operator-energy estimate; refusing NaN-to-one capture.")
    ratio=numerator/trace_estimate
    if not np.isfinite(ratio): raise RuntimeError("Non-finite operator capture.")
    return float(min(1.0,ratio))

def validate_task_frame(df,expected_ids,label):
    expected=tuple(map(int,expected_ids))
    if len(df)!=len(expected) or "replicate_id" not in df or "task_hash" not in df:
        raise RuntimeError(f"{label}: incomplete task coverage.")
    ids=df["replicate_id"].to_numpy(dtype=float)
    if not np.isfinite(ids).all() or not np.equal(ids,np.floor(ids)).all() or tuple(sorted(map(int,ids)))!=tuple(sorted(expected)):
        raise RuntimeError(f"{label}: unexpected/duplicate task IDs.")
    if df["task_hash"].isna().any() or df["task_hash"].nunique()!=len(expected):
        raise RuntimeError(f"{label}: missing/duplicate task identity.")
    numeric=df.select_dtypes(include=[np.number])
    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise RuntimeError(f"{label}: non-finite task measurement.")

def bootstrap_ci(vals,n=None,seed=0):
    x=_finite_vector(vals,"statistical input")

    n=int(CORE.bootstrap_n if n is None else n)
    rng=np.random.default_rng(seed)
    idx=rng.integers(0,len(x),size=(n,len(x)))
    means=x[idx].mean(1)
    return [float(np.quantile(means,.025)),float(np.quantile(means,.975))]

def mean_ci_record(vals,seed=0):
    x=_finite_vector(vals,"statistical input")
    return {"n":int(len(x)),"mean":float(x.mean()) if len(x) else float("nan"),
            "median":float(np.median(x)) if len(x) else float("nan"),
            "ci95":bootstrap_ci(x,seed=seed) if len(x) else [float("nan"),float("nan")]}

def one_sided_signflip_p(vals,seed=0):
    x=_finite_vector(vals,"statistical input")

    obs=float(x.mean())
    if len(x)<=20:
        sums=[]
        for mask in range(1<<len(x)):
            signs=np.array([1.0 if (mask>>j)&1 else -1.0 for j in range(len(x))])
            sums.append(float((x*signs).mean()))
        return float(np.mean(np.asarray(sums)>=obs))
    rng=np.random.default_rng(seed); draws=int(CORE.signflip_mc_draws)
    ge=0
    for _ in range(draws):
        signs=rng.choice(np.array([-1.0,1.0]),size=len(x))
        ge+=int(float((x*signs).mean())>=obs)
    return float((ge+1)/(draws+1))

def resolve_table_path(stem):
    stem=Path(stem); pq=Path(str(stem)+".parquet"); cv=Path(str(stem)+".csv")
    if pq.exists(): return pq
    if cv.exists(): return cv
    raise FileNotFoundError(stem)

BASE=ROOT/("SMOKE" if PROFILE=="smoke" else "RESEARCH")
BASE.mkdir(parents=True,exist_ok=True)
for d in ["manifests","discovery","confirmation","locks","logs","exports"]:
    (BASE/d).mkdir(parents=True,exist_ok=True)
def initialize_campaign_manifests():
    ident={"project":PROJECT,"version":VERSION,"profile":PROFILE,"protocol_hash":PROTOCOL_HASH,
           "scientific_code_hash":SCIENTIFIC_CODE_HASH,"env_compat_hash":ENV_COMPAT_HASH}
    ip=BASE/"manifests"/"campaign_identity.json"
    if ip.exists() and json.loads(ip.read_text())!=ident:
        raise RuntimeError("Campaign identity mismatch; existing provenance was not overwritten. Use the matching notebook/environment or a fresh campaign root.")
    # Also protect a directory created by an earlier implementation without campaign_identity.json.
    candidates=list((BASE/"locks").glob("*.json"))+list((BASE/"discovery").glob("rep_*_commit.json"))+list((BASE/"confirmation").glob("rep_*_commit.json"))
    oldscientific=BASE/"manifests"/"scientific_code_identity.json"
    if oldscientific.exists(): candidates.append(oldscientific)
    for p in candidates:
        x=json.loads(p.read_text())
        for key in ("protocol_hash","scientific_code_hash","env_compat_hash"):
            actual=x.get("sha256") if key=="scientific_code_hash" and p==oldscientific else x.get(key)
            if actual!=ident[key]: raise RuntimeError(f"Existing {p.name} identity mismatch; refusing manifest overwrite.")
    atomic_json(ip,ident)
    atomic_json(BASE/"manifests"/"environment.json",ENV)
    atomic_json(BASE/"manifests"/"core_config.json",asdict(CORE))
    atomic_json(BASE/"manifests"/"protocol_payload.json",PROTOCOL_PAYLOAD)
    atomic_json(BASE/"manifests"/"preregistered_analysis.json",PREREG)
    atomic_json(BASE/"manifests"/"parent_evidence.json",PARENT_EVIDENCE)
    atomic_json(BASE/"manifests"/"implementation_revision.json",{"revision":IMPLEMENTATION_REVISION})

def validate_temporal_frame(tdf,rep,label):
    if len(tdf)!=len(CORE.temporal_fracs) or "replicate_id" not in tdf or not np.equal(tdf["replicate_id"].to_numpy(float),int(rep)).all():
        raise RuntimeError(f"{label}: temporal task coverage mismatch for {rep}.")
    if not np.isfinite(tdf.select_dtypes(include=[np.number]).to_numpy(float)).all():
        raise RuntimeError(f"{label}: non-finite temporal measurement for {rep}.")
    if tuple(sorted(tdf["time_frac"].to_numpy(float)))!=tuple(sorted(CORE.temporal_fracs)):
        raise RuntimeError(f"{label}: temporal checkpoint mismatch for {rep}.")
    for _,r in tdf.iterrows():
        if float(r["checkpoint_step"])!=checkpoint_step_for(r["time_frac"],CORE.train_steps) or float(r["donor_step"])!=donor_step_for(r["time_frac"],CORE.train_steps):
            raise RuntimeError(f"{label}: temporal step mismatch for {rep}.")

def verify_task_tables(rep,gp,tp,commit,cohort,selected_k=None):
    if type(commit.get("replicate_id")) is not int or commit["replicate_id"]!=int(rep): raise RuntimeError(f"{cohort}: commit task-ID mismatch for {rep}.")
    gdf=load_table(gp); tdf=load_table(tp)
    validate_task_frame(gdf,(int(rep),),cohort)
    grow=gdf.iloc[0]
    for key,expected in (("protocol_hash",PROTOCOL_HASH),("scientific_code_hash",SCIENTIFIC_CODE_HASH),("env_compat_hash",ENV_COMPAT_HASH),("task_hash",commit.get("task_hash"))):
        if str(grow.get(key))!=str(expected): raise RuntimeError(f"{cohort}: geometry {key} mismatch for {rep}.")
    if len(tdf)!=len(CORE.temporal_fracs) or "replicate_id" not in tdf or not np.equal(tdf["replicate_id"].to_numpy(float),int(rep)).all():
        raise RuntimeError(f"{cohort}: temporal task coverage mismatch for {rep}.")
    numeric=tdf.select_dtypes(include=[np.number])
    if not np.isfinite(numeric.to_numpy(float)).all(): raise RuntimeError(f"{cohort}: non-finite temporal table for {rep}.")
    fracs=tdf["time_frac"].to_numpy(float)
    if tuple(sorted(fracs))!=tuple(sorted(CORE.temporal_fracs)): raise RuntimeError(f"{cohort}: temporal checkpoint mismatch for {rep}.")
    for _,r in tdf.iterrows():
        if float(r["checkpoint_step"])!=checkpoint_step_for(r["time_frac"],CORE.train_steps) or float(r["donor_step"])!=donor_step_for(r["time_frac"],CORE.train_steps):
            raise RuntimeError(f"{cohort}: temporal step mismatch for {rep}.")
    if selected_k is not None and int(grow.get("selected_k",-1))!=int(selected_k):
        raise RuntimeError(f"{cohort}: geometry rank mismatch for {rep}.")
    if cohort=="confirmation": verify_causal_endpoints(rep,selected_k,grow,commit)


In [ ]:
# 3. Fresh latent task generator (frozen from Stage06 controlled system)
@dataclass(frozen=True)
class SuiteSpec:
    max_K:int=48
    image_size:int=32
    n_classes:int=4
    train_n:int=4096
    direction_n:int=192
    calibration_n:int=192
    response_n:int=192
    evaluation_n:int=768
    sealed_n:int=768
    seed:int=0

ROLE_NAMES=('train','direction','calibration','response','evaluation','sealed')

def spatial_basis(max_K,S,seed):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[-1:1:complex(S),-1:1:complex(S)]; out=[]
    for _ in range(max_K):
        cx,cy=rng.uniform(-.75,.75,2); sx,sy=rng.uniform(.15,.55,2); f=rng.uniform(.5,3.0); a=rng.uniform(0,2*np.pi)
        xr=np.cos(a)*xx+np.sin(a)*yy
        b=np.exp(-((xx-cx)**2/(2*sx*sx)+(yy-cy)**2/(2*sy*sy)))*np.cos(np.pi*f*xr)
        b=b/(np.sqrt(np.mean(b*b))+1e-8); out.append(b.astype('float32'))
    return torch.tensor(np.stack(out),dtype=torch.float32)

def make_suite(spec:SuiteSpec):
    Z={}
    sizes={r:getattr(spec,f'{r}_n') for r in ROLE_NAMES}
    for role,n in sizes.items():
        g=torch.Generator().manual_seed(stable_seed('suite_z',spec.seed,role))
        # dimension-major generation makes first k coordinates invariant as max_K changes.
        Z[role]=torch.randn(spec.max_K,n,generator=g).T.contiguous()
    basis=spatial_basis(spec.max_K,spec.image_size,stable_seed('suite_basis',spec.seed))
    # Fixed odd nonlinear coefficient bank for nested 4-class teacher.
    rng=np.random.default_rng(stable_seed('label_coeff',spec.seed))
    coeff=torch.tensor(rng.uniform(0.7,1.5,size=spec.max_K),dtype=torch.float32)
    freq=torch.tensor(rng.uniform(0.7,2.0,size=spec.max_K),dtype=torch.float32)
    suite={'spec':asdict(spec),'Z':Z,'basis':basis,'coeff':coeff,'freq':freq}
    suite['suite_hash']=hash_tree({'spec':suite['spec'],'basis':basis,'coeff':coeff,'freq':freq,'Z':Z})
    return suite

def nested_clean_labels(z,r,coeff,freq):
    assert r>=2 and r<=z.shape[1]
    # Two nested coordinate streams define two balanced-ish bits -> 4 classes.
    ia=torch.arange(0,r,2); ib=torch.arange(1,r,2)
    def score(idx):
        if len(idx)==0:return torch.zeros(len(z))
        zz=z[:,idx]; c=coeff[idx]; f=freq[idx]
        term=torch.tanh(c*zz)+0.25*torch.sin(f*zz)
        return term.sum(1)/math.sqrt(len(idx))
    a=score(ia); b=score(ib)
    return ((a>0).long()*2+(b>0).long()), torch.minimum(a.abs(),b.abs())

def nested_noisy_labels(clean,eta,seed,role):
    if eta<=0:return clean.clone(),torch.zeros(len(clean),dtype=torch.bool)
    g=torch.Generator().manual_seed(stable_seed('label_noise',seed,role))
    u=torch.rand(len(clean),generator=g); mask=u<eta
    # Replacement is guaranteed to differ from clean label.
    off=1+torch.randint(0,3,(len(clean),),generator=g)
    noisy=(clean+off)%4
    return torch.where(mask,noisy,clean),mask

def render_fixedK(z,basis,K):
    raw=torch.einsum('nk,khw->nhw',z[:,:K],basis[:K])/math.sqrt(K)
    return torch.tanh(0.85*raw).unsqueeze(1)

def render_nuisanceK(z,basis,K,r,nuisance_gain=1.0):
    assert K>r>=2
    sig=torch.einsum('nk,khw->nhw',z[:,:r],basis[:r])/math.sqrt(r)
    nui=torch.einsum('nk,khw->nhw',z[:,r:K],basis[r:K])/math.sqrt(K-r)
    raw=(sig+nuisance_gain*nui)/math.sqrt(1+nuisance_gain**2)
    return torch.tanh(0.85*raw).unsqueeze(1)

def make_task_variant(suite,K,r,eta=0.0,mode='fixedK',replicate_id=0):
    assert 2<=r<=K<=suite['spec']['max_K']
    public={}; sealed={}; clean_hashes={}; noise_masks={}
    for role in ROLE_NAMES:
        z=suite['Z'][role]
        X=render_nuisanceK(z,suite['basis'],K,r) if mode=='nuisanceK' else render_fixedK(z,suite['basis'],K)
        y_clean,margin=nested_clean_labels(z,r,suite['coeff'],suite['freq'])
        # Noise changes the training objective, not the primary measurement operator.
        # The direction role stores a noisy objective label separately so conventional objective-gradient
        # diagnostics can still be measured without changing the clean-reference causal intervention.
        if role in {'train','direction'}:
            y_objective,mask=nested_noisy_labels(y_clean,eta,suite['spec']['seed'],role)
        else:
            y_objective=y_clean.clone(); mask=torch.zeros(len(y_clean),dtype=torch.bool)
        y_primary=y_objective if role=='train' else y_clean
        item={'X':X.contiguous(),'y':y_primary.contiguous(),'y_clean':y_clean.contiguous(),'y_objective':y_objective.contiguous(),'teacher_margin':margin.contiguous()}
        if role=='sealed': sealed=item
        else: public[role]=item
        clean_hashes[role]=hash_tree(y_clean); noise_masks[role]=mask
    spec={'K':K,'r':r,'label_noise':float(eta),'mode':mode,'n_classes':4,'replicate_id':int(replicate_id),'suite_hash':suite['suite_hash'],'suite_seed':suite['spec']['seed'],'in_ch':1}
    public_hash=hash_tree({'spec':spec,'public':public})
    sealed_hash=hash_tree({'spec':spec,'sealed':sealed})
    task={'spec':spec,'roles':public,'task_hash':public_hash,'sealed_hash':sealed_hash,
          'clean_label_hashes':clean_hashes,'noise_masks':noise_masks}
    return task,sealed

def get_sealed(sealed_bundle):
    if not bool(int(os.environ.get('NEXUS_UNLOCK_SEALED','0'))): raise RuntimeError('Sealed holdout is locked. Set NEXUS_UNLOCK_SEALED=1 only after analysis is frozen.')
    return sealed_bundle


In [ ]:
# 4. Architecture registry (Stage07 primary system = ResNet18-GN scale 2)

def _gn(c):
    g=min(32,max(1,c//8))
    while c%g!=0:g-=1
    return nn.GroupNorm(g,c)

class ConvCalib(nn.Module):
    def __init__(self,n_classes=4,in_ch=1,w=16):
        super().__init__(); self.stem=nn.Conv2d(in_ch,w,3,padding=1); self.b1=nn.Sequential(nn.Conv2d(w,2*w,3,stride=2,padding=1),nn.GELU()); self.b2=nn.Sequential(nn.Conv2d(2*w,4*w,3,stride=2,padding=1),nn.GELU()); self.pool=nn.AdaptiveAvgPool2d(1); self.head=nn.Linear(4*w,n_classes)
    def forward(self,x,return_features=False):
        f=[]; x=F.gelu(self.stem(x));f.append(x.mean((-2,-1)));x=self.b1(x);f.append(x.mean((-2,-1)));x=self.b2(x);f.append(x.mean((-2,-1)));y=self.head(self.pool(x).flatten(1));return (y,f) if return_features else y
    def intervention_targets(self):return ['stem','b1','b2','head']
    def target_module(self,n):return getattr(self,n)

class BasicBlockGN(nn.Module):
    expansion=1
    def __init__(self,cin,cout,stride=1):
        super().__init__(); self.c1=nn.Conv2d(cin,cout,3,stride=stride,padding=1,bias=False);self.n1=_gn(cout);self.c2=nn.Conv2d(cout,cout,3,padding=1,bias=False);self.n2=_gn(cout)
        self.skip=nn.Identity() if stride==1 and cin==cout else nn.Sequential(nn.Conv2d(cin,cout,1,stride=stride,bias=False),_gn(cout))
    def forward(self,x):
        y=F.gelu(self.n1(self.c1(x)));y=self.n2(self.c2(y));return F.gelu(y+self.skip(x))

class ResNet18GN(nn.Module):
    def __init__(self,n_classes=4,in_ch=1,base=32):
        super().__init__();self.stem=nn.Sequential(nn.Conv2d(in_ch,base,3,padding=1,bias=False),_gn(base),nn.GELU())
        self.layer1=self._layer(base,base,2,1);self.layer2=self._layer(base,2*base,2,2);self.layer3=self._layer(2*base,4*base,2,2);self.layer4=self._layer(4*base,8*base,2,2)
        self.pool=nn.AdaptiveAvgPool2d(1);self.head=nn.Linear(8*base,n_classes)
    def _layer(self,cin,cout,n,stride):return nn.Sequential(BasicBlockGN(cin,cout,stride),*[BasicBlockGN(cout,cout,1) for _ in range(n-1)])
    def forward(self,x,return_features=False):
        f=[];x=self.stem(x);f.append(x.mean((-2,-1)));x=self.layer1(x);f.append(x.mean((-2,-1)));x=self.layer2(x);f.append(x.mean((-2,-1)));x=self.layer3(x);f.append(x.mean((-2,-1)));x=self.layer4(x);f.append(x.mean((-2,-1)));z=self.pool(x).flatten(1);y=self.head(z);return (y,f) if return_features else y
    def intervention_targets(self):return ['stem','layer1','layer2','layer3','layer4','head']
    def target_module(self,n):return getattr(self,n)

class ViT(nn.Module):
    def __init__(self,n_classes=4,in_ch=1,dim=128,depth=4,heads=4,patch=4):
        super().__init__();self.patch=nn.Conv2d(in_ch,dim,patch,patch);nt=(32//patch)**2;self.cls=nn.Parameter(torch.zeros(1,1,dim));self.pos=nn.Parameter(torch.randn(1,nt+1,dim)*0.02)
        layer=nn.TransformerEncoderLayer(dim,heads,4*dim,dropout=0.0,activation='gelu',batch_first=True,norm_first=True)
        self.blocks=nn.ModuleList([copy.deepcopy(layer) for _ in range(depth)]);self.norm=nn.LayerNorm(dim);self.head=nn.Linear(dim,n_classes)
    def forward(self,x,return_features=False):
        x=self.patch(x).flatten(2).transpose(1,2);x=torch.cat([self.cls.expand(x.size(0),-1,-1),x],1)+self.pos[:,:x.size(1)+1];f=[]
        for b in self.blocks:x=b(x);f.append(x[:,0])
        z=self.norm(x[:,0]);y=self.head(z);return (y,f) if return_features else y
    def intervention_targets(self):return ['patch']+[f'blocks.{i}' for i in range(len(self.blocks))]+['head']
    def target_module(self,n):
        if n=='patch':return self.patch
        if n=='head':return self.head
        if n.startswith('blocks.'):return self.blocks[int(n.split('.')[1])]
        raise KeyError(n)

def build_model(family,n_classes,scale=1,seed=0,in_ch=1):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(stable_seed('model_init',family,scale,seed,in_ch,n_classes))
        if family=='calib':m=ConvCalib(n_classes,in_ch,w=16*scale)
        elif family=='resnet':m=ResNet18GN(n_classes,in_ch,base=16*scale)
        elif family=='vit':
            presets={1:(96,4,4),2:(192,6,6),3:(256,8,8),4:(384,8,8)};dim,depth,heads=presets[int(scale)];m=ViT(n_classes,in_ch,dim,depth,heads)
        else:raise ValueError(family)
    return m.to(DEVICE)

def parameter_count(m):return sum(p.numel() for p in m.parameters())

for fam,sc in [('calib',1),('resnet',1),('resnet',2),('resnet',4),('vit',1),('vit',2),('vit',4)]:
    m=build_model(fam,4,sc,0);print(f'{fam} scale={sc}: {parameter_count(m):,} params');del m
if torch.cuda.is_available():torch.cuda.empty_cache()


In [ ]:

# 5. Deterministic trainer and checkpoint bank
@dataclass(frozen=True)
class TrainSpec:
    steps:int; batch_size:int; lr:float; weight_decay:float; beta1:float; beta2:float; eps:float
    amsgrad:bool; maximize:bool; foreach:bool; capturable:bool; differentiable:bool; fused:bool

def s7_train_spec():
    return TrainSpec(CORE.train_steps,CORE.batch_size,CORE.lr,CORE.weight_decay,CORE.adam_beta1,CORE.adam_beta2,
                     CORE.adam_eps,CORE.adam_amsgrad,CORE.adam_maximize,CORE.adam_foreach,CORE.adam_capturable,
                     CORE.adam_differentiable,CORE.adam_fused)

def resolved_checkpoints(steps):
    fracs={0.0,1.0,CORE.recipient_frac}
    for f in CORE.temporal_fracs:
        fracs.add(float(f)); fracs.add(max(0.0,float(f)-CORE.donor_lag_frac))
    return tuple(sorted({int(round(f*steps)) for f in fracs}))

def paired_stream_id(task): return str(task["spec"]["suite_hash"])

def make_schedule(n,batch_size,steps,seed):
    assert n>=batch_size
    needed=steps+max(CORE.future_horizons)+2
    batches=[]; g=torch.Generator().manual_seed(seed); usable=(n//batch_size)*batch_size
    while len(batches)<needed:
        p=torch.randperm(n,generator=g)[:usable].reshape(-1,batch_size)
        batches.extend([row.clone() for row in p])
    cpu=torch.stack(batches[:needed])
    return cpu.to(DEVICE),hash_tree(cpu)

def make_optimizer(m,ts):
    return torch.optim.AdamW(m.parameters(),lr=ts.lr,betas=(ts.beta1,ts.beta2),eps=ts.eps,weight_decay=ts.weight_decay,
        amsgrad=ts.amsgrad,maximize=ts.maximize,foreach=ts.foreach,capturable=ts.capturable,differentiable=ts.differentiable,fused=ts.fused)

def optimizer_defaults_record(o):
    out={}
    for k,v in o.defaults.items():
        if isinstance(v,(str,int,float,bool,type(None))): out[k]=v
        elif isinstance(v,tuple): out[k]=list(v)
        else: out[k]=str(v)
    return out

def model_hash(m): return hash_tree(tree_to_cpu(m.state_dict()))
def optimizer_hash(o): return hash_tree(tree_to_cpu(o.state_dict()))

def step_rng_seed(stream_id,family,scale,model_seed,absolute_step):
    return stable_seed("step_rng",stream_id,model_seed,absolute_step)

def set_step_rng(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def prepare_device_task(task):
    out={"spec":task["spec"],"task_hash":task["task_hash"],"sealed_hash":task["sealed_hash"],"roles":{}}
    for role,d in task["roles"].items():
        out["roles"][role]={k:(v.to(DEVICE) if torch.is_tensor(v) else v) for k,v in d.items()}
    return out

def train_step(m,o,dtask,idx,abs_step,family,scale,model_seed):
    set_step_rng(step_rng_seed(paired_stream_id(dtask),family,scale,model_seed,abs_step))
    m.train(); X=dtask["roles"]["train"]["X"]; y=dtask["roles"]["train"]["y"]
    o.zero_grad(set_to_none=True); loss=F.cross_entropy(m(X[idx]),y[idx]); loss.backward(); o.step()
    return float(loss.detach().cpu())

def snapshot(m,o,step,meta):
    return {"step":int(step),"model":tree_to_cpu(m.state_dict()),"optimizer":tree_to_cpu(o.state_dict()),
            "model_hash":model_hash(m),"optimizer_hash":optimizer_hash(o),**copy.deepcopy(meta)}

def load_snapshot(task,family,scale,model_seed,ts,snap):
    assert snap["task_hash"]==task["task_hash"] and snap["core_config_hash"]==CONFIG_HASH and snap["protocol_hash"]==PROTOCOL_HASH
    assert snap.get("env_compat_hash")==ENV_COMPAT_HASH
    m=build_model(family,task["spec"]["n_classes"],scale,model_seed,int(task["spec"].get("in_ch",1))); m.load_state_dict(copy.deepcopy(snap["model"]))
    o=make_optimizer(m,ts); o.load_state_dict(copy.deepcopy(snap["optimizer"]))
    for st in o.state.values():
        for k,v in st.items():
            if torch.is_tensor(v): st[k]=v.to(DEVICE)
    assert model_hash(m)==snap["model_hash"] and optimizer_hash(o)==snap["optimizer_hash"]
    return m,o

def build_checkpoint_bank(task,family,scale,model_seed,ts):
    dt=prepare_device_task(task); stream=paired_stream_id(task)
    sch,shash=make_schedule(len(task["roles"]["train"]["X"]),ts.batch_size,ts.steps,stable_seed("schedule",stream,model_seed,asdict(ts)))
    m=build_model(family,task["spec"]["n_classes"],scale,model_seed,int(task["spec"].get("in_ch",1))); o=make_optimizer(m,ts)
    cps=resolved_checkpoints(ts.steps); bank={}; model_n=parameter_count(m)
    meta={"family":family,"scale":scale,"model_seed":model_seed,"task_hash":task["task_hash"],"paired_stream_id":stream,
          "schedule_hash":shash,"core_config_hash":CONFIG_HASH,"protocol_hash":PROTOCOL_HASH,"env_compat_hash":ENV_COMPAT_HASH,
          "train_spec_hash":hashlib.sha256(json.dumps(asdict(ts),sort_keys=True).encode()).hexdigest(),
          "train_spec":asdict(ts),"optimizer_defaults":optimizer_defaults_record(o),"model_parameter_count":int(model_n)}
    for st in range(ts.steps+1):
        if st in cps: bank[st]=snapshot(m,o,st,meta)
        if st<ts.steps: train_step(m,o,dt,sch[st],st,family,scale,model_seed)
    final_eval=evaluate_public(m,dt)
    del m,o; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return bank,sch,dt,final_eval


In [ ]:
# 6. Evaluation utilities
def forward_bundle(m,X,y=None,return_features=True):
    was=m.training;m.eval()
    with torch.no_grad():
        out=m(X,return_features=return_features) if return_features else m(X)
    if return_features:logits,feats=out
    else:logits=out;feats=[]
    probs=logits.softmax(-1)
    ret={'logits':logits.detach(),'features':[f.detach() for f in feats],
         'entropy':float((-(probs*torch.log(probs.clamp_min(1e-12))).sum(-1).mean()).cpu())}
    if y is not None:
        true=logits.gather(1,y[:,None]).squeeze(1);tmp=logits.clone();tmp.scatter_(1,y[:,None],float('-inf'))
        ret.update({'loss':float(F.cross_entropy(logits,y).cpu()),
                    'acc':float((logits.argmax(1)==y).float().mean().cpu()),
                    'margin':float((true-tmp.max(1).values).mean().cpu())})
    m.train(was);return ret

def evaluate_public(m,dt):
    d=dt['roles']['evaluation'];return forward_bundle(m,d['X'],d['y_clean'],False)

def tuple_norm(gs):return math.sqrt(sum(float((g.detach().float()**2).sum().cpu()) for g in gs if g is not None))

def optimizer_moment_norms(o):
    a1=a2=0.0
    for st in o.state.values():
        if 'exp_avg' in st:a1+=float((st['exp_avg'].detach().float()**2).sum().cpu())
        if 'exp_avg_sq' in st:a2+=float((st['exp_avg_sq'].detach().float()**2).sum().cpu())
    return math.sqrt(a1),math.sqrt(a2)

def optimizer_preconditioned_norm(o,eps=None):
    eps=float(CORE.adam_eps if eps is None else eps);s=0.0
    for st in o.state.values():
        if 'exp_avg' in st and 'exp_avg_sq' in st:
            u=st['exp_avg'].detach().float()/(st['exp_avg_sq'].detach().float().sqrt()+eps)
            s+=float((u*u).sum().cpu())
    return math.sqrt(s)

def optimizer_step_stats(o):
    vals=[]
    for st in o.state.values():
        if 'step' in st:
            x=st['step']
            vals.append(float(x.detach().cpu().item()) if torch.is_tensor(x) else float(x))
    if not vals:return (0.0,0.0)
    return (float(min(vals)),float(max(vals)))

def gradient_tuple(m,X,y):
    was=m.training;m.eval();m.zero_grad(set_to_none=True);loss=F.cross_entropy(m(X),y)
    g=torch.autograd.grad(loss,tuple(m.parameters()),retain_graph=False,create_graph=False);m.train(was);return g

In [ ]:

# 7. Hidden optimizer-memory axes and exact differentiable one-step control operator
def _named_param_map(m): return dict(m.named_parameters())

def _axis_norm(a):
    return math.sqrt(sum(float((v.detach().float()**2).sum().cpu()) for v in a.values()))

def _axis_dot(a,b):
    if set(a)!=set(b): raise RuntimeError("Axis parameter-key mismatch.")
    keys=sorted(a)
    return sum(float((a[k].detach().float()*b[k].detach().float()).sum().cpu()) for k in keys)

def _axis_scale(a,s): return {k:v*float(s) for k,v in a.items()}
def _axis_add(a,b,sa=1.0,sb=1.0): return {k:float(sa)*a[k]+float(sb)*b[k] for k in a}
def _axis_cos(a,b):
    return _axis_dot(a,b)/max(_axis_norm(a)*_axis_norm(b),1e-30)

def _axis_sub_projection(a,b):
    den=_axis_dot(b,b)
    if den<=1e-30: return {k:v.clone() for k,v in a.items()}
    c=_axis_dot(a,b)/den
    return {k:a[k]-c*b[k] for k in a}

def _axis_normalize(a):
    n=_axis_norm(a)
    if n<=CORE.min_axis_norm: raise RuntimeError("Degenerate axis.")
    return _axis_scale(a,1.0/n)

def _axis_linear_combination(basis,weights):
    out={k:torch.zeros_like(basis[0][k]) for k in basis[0]}
    for a,w in zip(basis,weights):
        for k in out: out[k].add_(a[k],alpha=float(w))
    return out

def _axis_project(a,basis):
    out={k:torch.zeros_like(v) for k,v in a.items()}
    for u in basis:
        c=_axis_dot(a,u)
        for k in out: out[k].add_(u[k],alpha=float(c))
    return out

def _axis_projection_fraction(a,basis):
    den=_axis_dot(a,a)
    if den<=1e-30: return float("nan")
    return float(sum(_axis_dot(a,u)**2 for u in basis)/den)

def _orthonormalize_axes(vectors,tol=1e-10):
    Q=[]
    for v0 in vectors:
        v={k:x.clone() for k,x in v0.items()}
        for _ in range(2):
            for q in Q:
                c=_axis_dot(v,q)
                for k in v: v[k].add_(q[k],alpha=-float(c))
        n=_axis_norm(v)
        if n>tol: Q.append(_axis_scale(v,1.0/n))
    return Q

def _history_axis(rec_m,rec_o,don_m,don_o):
    rn=dict(rec_m.named_parameters()); dn=dict(don_m.named_parameters()); out={}
    for name,p in rn.items():
        sr=rec_o.state.get(p,{})
        if "exp_avg" not in sr: continue
        pd=dn[name]; sd=don_o.state.get(pd,{})
        md=sd.get("exp_avg",torch.zeros_like(sr["exp_avg"]))
        out[name]=sr["exp_avg"].detach().float()-md.detach().float().to(sr["exp_avg"].device)
    if _axis_norm(out)<=CORE.min_axis_norm: raise RuntimeError("Historical axis is degenerate.")
    return out

def _match_tensor_norms(raw,reference,orthogonalize_to=None):
    out={}
    for k,h in reference.items():
        v=raw[k].detach().float().clone()
        if orthogonalize_to is not None:
            b=orthogonalize_to[k].detach().float()
            den=float((b*b).sum().cpu())
            if den>1e-30: v=v-(float((v*b).sum().cpu())/den)*b
        hn=float(torch.linalg.vector_norm(h).cpu()); vn=float(torch.linalg.vector_norm(v).cpu())
        out[k]=torch.zeros_like(h) if hn<=1e-30 or vn<=1e-30 else v*(hn/vn)
    return out

def build_strong_controls(m,o,history,grad_map,precond_map,replicate_id,n_random):
    controls={}
    for j in range(int(n_random)):
        raw={}
        for k,h in history.items():
            g=torch.Generator(device="cpu").manual_seed(stable_seed("S7_tensor_random",replicate_id,j,k))
            raw[k]=torch.randn(tuple(h.shape),generator=g,dtype=torch.float32).to(h.device)
        controls[f"tensor_random_{j:02d}"]=_match_tensor_norms(raw,history,orthogonalize_to=history)
    sign={}
    perm={}
    for k,h in history.items():
        g=torch.Generator(device="cpu").manual_seed(stable_seed("S7_signshuffle",replicate_id,k))
        s=(torch.randint(0,2,tuple(h.shape),generator=g,dtype=torch.int8).float()*2-1).to(h.device)
        sign[k]=h*s
        gp=torch.Generator(device="cpu").manual_seed(stable_seed("S7_permute",replicate_id,k))
        idx=torch.randperm(h.numel(),generator=gp)
        perm[k]=h.flatten()[idx.to(h.device)].reshape_as(h)
    controls["sign_shuffle"]=_match_tensor_norms(sign,history)
    controls["within_tensor_permute"]=_match_tensor_norms(perm,history)
    controls["current_gradient"]=_match_tensor_norms(grad_map,history)
    controls["preconditioned_gradient"]=_match_tensor_norms(precond_map,history)
    return controls

def _optimizer_state_maps(m,o):
    m1={}; v={}; stepvals=[]
    for name,p in m.named_parameters():
        st=o.state.get(p,{})
        if "exp_avg" in st:
            m1[name]=st["exp_avg"].detach().float()
            v[name]=st["exp_avg_sq"].detach().float()
            sv=st.get("step",0.0); stepvals.append(float(sv.detach().cpu()) if torch.is_tensor(sv) else float(sv))
    if not stepvals: raise RuntimeError("Optimizer state not initialized at control checkpoint.")
    if max(stepvals)-min(stepvals)>1e-6: raise RuntimeError("Unexpected non-uniform AdamW step counters.")
    return m1,v,int(round(stepvals[0]))

def build_control_operator(task,dt,family,scale,model_seed,ts,snap,sch,checkpoint_step,anchor_role="calibration"):
    m,o=load_snapshot(task,family,scale,model_seed,ts,snap)
    m.train()
    idx=sch[int(checkpoint_step)]
    Xtr=dt["roles"]["train"]["X"][idx]; ytr=dt["roles"]["train"]["y"][idx]
    params=dict(m.named_parameters()); buffers=dict(m.named_buffers())
    plist=[params[n] for n in params]
    o.zero_grad(set_to_none=True)
    loss=F.cross_entropy(m(Xtr),ytr)
    grads=torch.autograd.grad(loss,plist,retain_graph=False,create_graph=False)
    grad_map={n:g.detach().float() for n,g in zip(params,grads)}
    m1,v,step0=_optimizer_state_maps(m,o)
    names=[n for n in params if n in m1]
    if set(names)!=set(m1): raise RuntimeError("Parameter/state name mismatch.")
    beta1,beta2=ts.beta1,ts.beta2; step1=step0+1
    bc1=1.0-beta1**step1; bc2=1.0-beta2**step1
    denom={n:(v[n]*beta2+(1-beta2)*grad_map[n].square()).sqrt()/math.sqrt(bc2)+ts.eps for n in names}
    precond_map={n:grad_map[n]/denom[n] for n in names}
    p0={n:params[n].detach() for n in params}
    g0={n:grad_map[n] for n in names}; m0={n:m1[n] for n in names}
    if anchor_role not in dt["roles"]: raise KeyError(anchor_role)
    Xanchor=dt["roles"][anchor_role]["X"][:min(CORE.control_anchor_n,len(dt["roles"][anchor_role]["X"]))]

    def op(delta):
        newp={}
        for n,p in p0.items():
            if n in m0:
                mn=beta1*(m0[n]+delta[n])+(1-beta1)*g0[n]
                upd=(ts.lr/bc1)*mn/denom[n]
                newp[n]=p*(1-ts.lr*ts.weight_decay)-upd
            else:
                newp[n]=p
        logits=functional_call(m,(newp,buffers),(Xanchor,))
        logits=logits-logits.mean(-1,keepdim=True)
        return logits.reshape(-1)

    zero={n:torch.zeros_like(m0[n]) for n in names}
    base=op(zero)
    # Exact next-step parameter tree for AdamW equivalence QA.
    manual={}
    for n,p in p0.items():
        if n in m0:
            mn=beta1*m0[n]+(1-beta1)*g0[n]
            manual[n]=p*(1-ts.lr*ts.weight_decay)-(ts.lr/bc1)*mn/denom[n]
        else: manual[n]=p
    ctx={"model":m,"optimizer":o,"op":op,"zero":zero,"base_output":base.detach(),"grad_map":{n:grad_map[n] for n in names},
         "precond_map":precond_map,"manual_next_params":manual,"names":names,"checkpoint_step":int(checkpoint_step),"anchor_role":anchor_role}
    return ctx

def operator_vjp(ctx,probe):
    _,pull=vjp(ctx["op"],ctx["zero"])
    g=pull(probe)[0]
    return {k:v.detach() for k,v in g.items()}

def operator_jvp(ctx,axis):
    _,t=jvp(ctx["op"],(ctx["zero"],),(axis,))
    return t.detach()

def operator_axis_potency(ctx,axis):
    n=_axis_norm(axis)
    if n<=CORE.min_axis_norm: return float("nan")
    y=operator_jvp(ctx,_axis_scale(axis,1.0/n))
    return float(torch.linalg.vector_norm(y).cpu())

def _rademacher_like_output(ref,seed,normalize=False):
    g=torch.Generator(device="cpu").manual_seed(seed)
    u=(torch.randint(0,2,tuple(ref.shape),generator=g,dtype=torch.int8).float()*2-1).to(ref.device)
    if normalize: u=u/max(float(torch.linalg.vector_norm(u).cpu()),1e-30)
    return u

def _random_axis_like(reference,seed):
    out={}
    for k,v in reference.items():
        g=torch.Generator(device="cpu").manual_seed(stable_seed("S7_right_space_pad",seed,k))
        out[k]=torch.randn(tuple(v.shape),generator=g,dtype=torch.float32).to(v.device)
    return out

def _normalized_control_range(vectors):
    normalized=[]
    for v in vectors:
        n=_axis_norm(v)
        if not np.isfinite(n): raise RuntimeError("Non-finite control-range vector.")
        if n>CORE.min_axis_norm: normalized.append(_axis_scale(v,1.0/n))
    return _orthonormalize_axes(normalized,tol=CORE.control_basis_tolerance)

def _complete_control_basis(Q,reference,replicate_id,iteration):
    pad_idx=0
    while len(Q)<max(CORE.rank_candidates):
        raw=_random_axis_like(reference,stable_seed("S7_range_pad_R4",replicate_id,iteration,pad_idx)); pad_idx+=1
        raw=_axis_normalize(raw)
        cand=_orthonormalize_axes(Q+[raw],tol=CORE.control_basis_tolerance)
        if len(cand)>len(Q): Q=cand
        if pad_idx>10*max(CORE.rank_candidates): raise RuntimeError("Unable to complete deterministic right-space challenger basis.")
    return Q

def estimate_local_control_geometry(ctx,replicate_id):
    # One reverse-mode linearization is reused for all output probes. This is both faster and
    # substantially more memory-stable than rebuilding the VJP graph for every probe.
    _,pull=vjp(ctx["op"],ctx["zero"])
    grads=[]
    for j in range(CORE.basis_probes):
        u=_rademacher_like_output(ctx["base_output"],stable_seed("S7_basis_output_probe",replicate_id,j),normalize=True)
        gg=pull(u)[0]
        grads.append({k:v.detach() for k,v in gg.items()})
    Q=_normalized_control_range(grads)
    del grads
    n_vjp_range_vectors=len(Q)
    Q=_complete_control_basis(Q,ctx["zero"],replicate_id,0)
    # Stabilized power/subspace iteration uses only the fit anchor, never history or evaluation.
    n_power_range_vectors=len(Q)
    for iteration in range(CORE.basis_power_iterations):
        vectors=[]
        for q in Q:
            y=operator_jvp(ctx,q)
            yn=float(torch.linalg.vector_norm(y.double()).cpu())
            if not np.isfinite(yn): raise RuntimeError("Non-finite power-iteration response.")
            if yn>CORE.min_axis_norm:
                gg=pull(y/yn)[0]; vectors.append({k:v.detach() for k,v in gg.items()})
        Q=_normalized_control_range(vectors)
        del vectors
        n_power_range_vectors=len(Q)
        Q=_complete_control_basis(Q,ctx["zero"],replicate_id,iteration+1)
    # Ritz ordering inside the augmented history-independent right-subspace.
    Y=[operator_jvp(ctx,q) for q in Q]
    G=torch.empty((len(Q),len(Q)),dtype=torch.float64)
    for i in range(len(Q)):
        for j in range(i,len(Q)):
            z=float(torch.dot(Y[i].double(),Y[j].double()).cpu()); G[i,j]=G[j,i]=z
    ev,W=torch.linalg.eigh(G)
    order=torch.argsort(ev,descending=True); ev=ev[order].clamp_min(0); W=W[:,order]
    U=[]
    for j in range(len(Q)):
        U.append(_axis_normalize(_axis_linear_combination(Q,W[:,j].cpu().numpy())))
    # Independent Hutchinson trace estimate of ||J||_F^2, using the same frozen linearization
    # but a disjoint deterministic probe namespace.
    tr=[]
    for j in range(CORE.trace_probes):
        u=_rademacher_like_output(ctx["base_output"],stable_seed("S7_trace_output_probe",replicate_id,j),normalize=False)
        gg=pull(u)[0]
        gg={k:v.detach() for k,v in gg.items()}
        tr.append(_axis_dot(gg,gg))
        del gg
    del pull
    trace_est=float(np.mean(tr))
    eigvals=[float(x) for x in ev.cpu().numpy()]
    captures={int(k):_validated_capture(sum(eigvals[:int(k)]),trace_est) for k in CORE.rank_candidates}
    return {"basis":U,"ritz_eigenvalues":eigvals,"trace_estimate":trace_est,"captures":captures,
            "n_vjp_range_vectors":int(n_vjp_range_vectors),"n_power_range_vectors":int(n_power_range_vectors),"power_iterations":CORE.basis_power_iterations,"n_total_basis_vectors":len(Q),"trace_probe_values":[float(x) for x in tr]}

def evaluate_subspace_transfer(ctx,basis,replicate_id):
    Y=[operator_jvp(ctx,u) for u in basis[:max(CORE.rank_candidates)]]
    prefix=[]; acc=0.0
    for y in Y:
        acc+=float(torch.dot(y.double(),y.double()).cpu()); prefix.append(acc)
    _,pull=vjp(ctx["op"],ctx["zero"])
    tr=[]
    for j in range(CORE.transfer_trace_probes):
        u=_rademacher_like_output(ctx["base_output"],stable_seed("S7_transfer_trace_probe",replicate_id,j),normalize=False)
        gg=pull(u)[0]; gg={k:v.detach() for k,v in gg.items()}
        tr.append(_axis_dot(gg,gg)); del gg
    del pull
    trace_est=float(np.mean(tr))
    captures={int(k):_validated_capture(prefix[int(k)-1],trace_est) for k in CORE.rank_candidates}
    return {"trace_estimate":trace_est,"captures":captures,"trace_probe_values":[float(x) for x in tr]}

def finite_history_locality(ctx,history,alpha):
    pred=operator_jvp(ctx,history)
    zp=ctx["op"](_axis_scale(history,+float(alpha))).detach()
    zm=ctx["op"](_axis_scale(history,-float(alpha))).detach()
    sec=(zp-zm)/(2.0*float(alpha))
    pn=float(torch.linalg.vector_norm(pred).cpu()); sn=float(torch.linalg.vector_norm(sec).cpu())
    cosine=float(torch.dot(pred.double(),sec.double()).cpu()/max(pn*sn,1e-30))
    drift=float(abs(sn-pn)/max(pn,1e-30))
    half_effect=(zp-zm)/2.0
    rmse=float(torch.linalg.vector_norm(half_effect).cpu()/math.sqrt(max(1,half_effect.numel())))
    passed=bool(cosine>=CORE.hidden_locality_min_secant_cosine and drift<=CORE.hidden_locality_max_relative_norm_drift and rmse>=CORE.causal_min_one_step_rmse)
    return {"finite_locality_cosine":cosine,"finite_locality_relative_norm_drift":drift,"full_one_step_rmse":rmse,"finite_locality_pass":passed}

def apply_m1_axis_surgery(m,o,axis,scale):
    nm=dict(m.named_parameters()); changed=0
    for name,d in axis.items():
        p=nm[name]; st=o.state.get(p,{})
        if "exp_avg" not in st: continue
        st["exp_avg"].add_(d.to(st["exp_avg"].device,dtype=st["exp_avg"].dtype),alpha=float(scale)); changed+=1
    if changed==0: raise RuntimeError("Surgery changed zero exp_avg tensors.")

def fixed_relative_scale(m,o,axis,relative):
    m1norm=optimizer_moment_norms(o)[0]; an=_axis_norm(axis)
    return float(relative*max(m1norm,1e-30)/max(an,1e-30))

def _centered_logits(m,X):
    was=m.training; m.eval()
    with torch.no_grad():
        z=m(X); z=z-z.mean(-1,keepdim=True)
    m.train(was)
    return z.detach().reshape(-1)

def finite_future_branch(task,dt,family,scale,model_seed,ts,snap,sch,axis,alpha,sign):
    m,o=load_snapshot(task,family,scale,model_seed,ts,snap)
    before_model=model_hash(m); Xresp=dt["roles"]["response"]["X"]; z0=_centered_logits(m,Xresp)
    if axis is not None and sign!=0: apply_m1_axis_surgery(m,o,axis,float(sign)*float(alpha))
    if model_hash(m)!=before_model: raise RuntimeError("m1 surgery changed model parameters at cut.")
    z0_after=_centered_logits(m,Xresp)
    inv=float(torch.max(torch.abs(z0_after-z0)).cpu())
    if inv!=0.0: raise RuntimeError(f"Function invariance failed at cut: {inv}")
    out={0:z0_after.cpu()}
    cp=int(snap["step"])
    for h in range(1,CORE.primary_future_horizon+1):
        train_step(m,o,dt,sch[cp+h-1],cp+h-1,family,scale,model_seed)
        if h in CORE.future_horizons: out[h]=_centered_logits(m,Xresp).cpu()
    del m,o; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out,inv


In [ ]:

# 8. Stage07 per-task measurement and confirmation statistics
def make_stage07_task(rep):
    spec=SuiteSpec(seed=stable_seed("S7_suite",rep))
    suite=make_suite(spec)
    task,_=make_task_variant(suite,K=32,r=4,eta=0.0,mode="fixedK",replicate_id=rep)
    return task

def donor_step_for(frac,steps):
    return int(round(max(0.0,float(frac)-CORE.donor_lag_frac)*steps))

def checkpoint_step_for(frac,steps): return int(round(float(frac)*steps))

def temporal_history_panel(task,dt,bank,sch,rep,ts):
    rows=[]
    for frac in CORE.temporal_fracs:
        cp=checkpoint_step_for(frac,ts.steps); dp=donor_step_for(frac,ts.steps)
        rm,ro=load_snapshot(task,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[cp])
        dm,do=load_snapshot(task,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[dp])
        hist=_history_axis(rm,ro,dm,do)
        ctx=build_control_operator(task,dt,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[cp],sch,cp)
        controls=build_strong_controls(rm,ro,hist,ctx["grad_map"],ctx["precond_map"],rep*1000+cp,CORE.tensor_random_controls)
        hp=operator_axis_potency(ctx,hist)
        rand=[operator_axis_potency(ctx,a) for k,a in controls.items() if k.startswith("tensor_random_")]
        rows.append({"replicate_id":int(rep),"time_frac":float(frac),"checkpoint_step":cp,"donor_step":dp,
                     "history_potency":hp,"median_tensor_random_potency":float(np.median(rand)),
                     "history_random_ratio":float(hp/max(np.median(rand),1e-30))})
        del rm,ro,dm,do,ctx,controls
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return rows

def causal_effect_metrics(vf,vp,vo):
    vectors=[v.detach().cpu().double().reshape(-1) for v in (vf,vp,vo)]
    if len({v.numel() for v in vectors})!=1 or any(not bool(torch.isfinite(v).all()) for v in vectors):
        raise RuntimeError("Invalid future causal-response vectors.")
    vf,vp,vo=vectors
    nf=float(torch.linalg.vector_norm(vf)); np_=float(torch.linalg.vector_norm(vp)); no=float(torch.linalg.vector_norm(vo))
    return {"full_norm":nf,"full_rmse":nf/math.sqrt(max(1,vf.numel())),
            "parallel_norm_ratio":np_/max(nf,1e-30),"perp_norm_ratio":no/max(nf,1e-30),
            "parallel_cosine":float(torch.dot(vp,vf))/max(np_*nf,1e-30),
            "additivity_error":float(torch.linalg.vector_norm(vf-vp-vo))/max(nf,1e-30)}

def causal_gate_from_records(ret,cos,perp,add):
    return bool(ret["mean"]>=CORE.causal_parallel_retention_mean_min and ret["ci95"][0]>=CORE.causal_parallel_retention_ci_lower_min and
                cos["ci95"][0]>=CORE.causal_parallel_cosine_ci_lower_min and perp["mean"]<=CORE.causal_perp_ratio_mean_max and
                perp["ci95"][1]<=CORE.causal_perp_ratio_ci_upper_max and add["mean"]<=CORE.causal_additivity_error_mean_max)

def save_causal_endpoints(rep,selected_k,alpha,baseline,branches):
    payload={"replicate_id":np.asarray(int(rep)),"selected_k":np.asarray(int(selected_k)),
             "alpha":np.asarray(float(alpha)),"protocol_hash":np.asarray(PROTOCOL_HASH),
             "scientific_code_hash":np.asarray(SCIENTIFIC_CODE_HASH)}
    for h,z in baseline.items(): payload[f"baseline_h{int(h)}"]=z.detach().cpu().numpy()
    for (name,sign),endpoints in branches.items():
        for h,z in endpoints.items(): payload[f"{name}_{'plus' if sign>0 else 'minus'}_h{int(h)}"]=z.detach().cpu().numpy()
    rel=f"confirmation/rep_{int(rep)}_causal_endpoints.npz"; path=BASE/rel; tmp=Path(str(path)+".tmp")
    with open(tmp,"wb") as f:
        np.savez_compressed(f,**payload); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)
    return {"causal_endpoints_path":rel,"causal_endpoints_sha256":file_sha256(path)}

def verify_causal_endpoints(rep,selected_k,row,commit):
    expected=f"confirmation/rep_{int(rep)}_causal_endpoints.npz"
    if row.get("causal_endpoints_path")!=expected or commit.get("causal_endpoints_path")!=expected:
        raise RuntimeError("Causal endpoint path mismatch.")
    path=BASE/expected; sha=file_sha256(path)
    if row.get("causal_endpoints_sha256")!=sha or commit.get("causal_endpoints_sha256")!=sha:
        raise RuntimeError("Causal endpoint hash mismatch.")
    horizons=(0,)+tuple(CORE.future_horizons)
    keys={"replicate_id","selected_k","alpha","protocol_hash","scientific_code_hash"}
    keys.update(f"baseline_h{h}" for h in horizons)
    keys.update(f"{name}_{sign}_h{h}" for name in ("history","parallel","perp") for sign in ("plus","minus") for h in horizons)
    with np.load(path,allow_pickle=False) as saved:
        if set(saved.files)!=keys: raise RuntimeError("Causal endpoint coverage mismatch.")
        if int(saved["replicate_id"])!=int(rep) or int(saved["selected_k"])!=int(selected_k): raise RuntimeError("Causal endpoint task/rank mismatch.")
        if str(saved["protocol_hash"])!=PROTOCOL_HASH or str(saved["scientific_code_hash"])!=SCIENTIFIC_CODE_HASH: raise RuntimeError("Causal endpoint provenance mismatch.")
        if float(saved["alpha"])!=float(row["causal_alpha"]): raise RuntimeError("Causal endpoint scale mismatch.")
        arrays=[saved[k] for k in keys if "_h" in k and k not in {"protocol_hash","scientific_code_hash"}]
        if len({a.shape for a in arrays})!=1 or any(a.ndim!=1 or not np.isfinite(a).all() for a in arrays):
            raise RuntimeError("Invalid causal endpoint arrays.")
        cut=saved["baseline_h0"]
        if any(not np.array_equal(saved[f"{name}_{sign}_h0"],cut) for name in ("history","parallel","perp") for sign in ("plus","minus")):
            raise RuntimeError("Saved causal endpoints violate cut-function invariance.")

def main_geometry_measurement(task,dt,bank,sch,rep,ts,selected_k=None,run_causal=False):
    cp=checkpoint_step_for(CORE.recipient_frac,ts.steps); dp=donor_step_for(CORE.recipient_frac,ts.steps)
    rm,ro=load_snapshot(task,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[cp])
    dm,do=load_snapshot(task,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[dp])
    hist=_history_axis(rm,ro,dm,do)
    ctx=build_control_operator(task,dt,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[cp],sch,cp,anchor_role="calibration")
    geom=estimate_local_control_geometry(ctx,rep)
    ctx_transfer=build_control_operator(task,dt,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[cp],sch,cp,anchor_role="evaluation")
    transfer=evaluate_subspace_transfer(ctx_transfer,geom["basis"],rep)
    controls=build_strong_controls(rm,ro,hist,ctx["grad_map"],ctx["precond_map"],rep,CORE.tensor_random_controls)

    row={"replicate_id":int(rep),"task_hash":str(task["task_hash"]),"recipient_acc":float(evaluate_public(rm,dt)["acc"]),
         "history_norm":_axis_norm(hist),"m1_norm":optimizer_moment_norms(ro)[0],
         "trace_fit":geom["trace_estimate"],"trace_transfer":transfer["trace_estimate"],
         "n_vjp_range_vectors":int(geom["n_vjp_range_vectors"]),"n_power_range_vectors":int(geom["n_power_range_vectors"]),"power_iterations":int(geom["power_iterations"]),"n_total_basis_vectors":int(geom["n_total_basis_vectors"])}
    for k,v in geom["captures"].items(): row[f"capture_fit_k{k}"]=float(v)
    for k,v in transfer["captures"].items(): row[f"capture_transfer_k{k}"]=float(v)

    hp_fit=operator_axis_potency(ctx,hist); hp_transfer=operator_axis_potency(ctx_transfer,hist)
    row["history_potency_fit"]=hp_fit; row["history_potency_transfer"]=hp_transfer
    for name,a in controls.items():
        row[f"align_{name}"]=_axis_projection_fraction(a,geom["basis"][:max(CORE.rank_candidates)])
        row[f"potency_fit_{name}"]=operator_axis_potency(ctx,a)
        row[f"potency_transfer_{name}"]=operator_axis_potency(ctx_transfer,a)
    # Candidate-rank history alignment is saved for every rank without choosing a result-dependent rank.
    for k in CORE.rank_candidates:
        row[f"history_alignment_k{k}"]=_axis_projection_fraction(hist,geom["basis"][:int(k)])

    if selected_k is not None:
        U=geom["basis"][:int(selected_k)]
        hpar=_axis_project(hist,U); hperp=_axis_add(hist,hpar,1.0,-1.0)
        row["selected_k"]=int(selected_k)
        row["history_alignment_selected"]=_axis_projection_fraction(hist,U)
        row["parallel_norm_fraction"]=_axis_norm(hpar)/max(_axis_norm(hist),1e-30)
        rand_align=[_axis_projection_fraction(a,U) for n,a in controls.items() if n.startswith("tensor_random_")]
        rand_pot=[row[f"potency_transfer_{n}"] for n in controls if n.startswith("tensor_random_")]
        row["median_random_alignment"]=float(np.median(rand_align))
        row["alignment_excess_vs_random"]=float(row["history_alignment_selected"]-np.median(rand_align))
        row["median_random_potency"]=float(np.median(rand_pot)); row["max_random_potency"]=float(np.max(rand_pot))
        row["history_random_potency_ratio"]=float(hp_transfer/max(np.median(rand_pot),1e-30))
        row["history_best_random_potency_ratio"]=float(hp_transfer/max(np.max(rand_pot),1e-30))
        row["history_signshuffle_potency_ratio"]=float(hp_transfer/max(row["potency_transfer_sign_shuffle"],1e-30))
        row["history_permute_potency_ratio"]=float(hp_transfer/max(row["potency_transfer_within_tensor_permute"],1e-30))
        row["history_gradient_potency_ratio"]=float(hp_transfer/max(row["potency_transfer_current_gradient"],1e-30))
        row["history_preconditioned_gradient_potency_ratio"]=float(hp_transfer/max(row["potency_transfer_preconditioned_gradient"],1e-30))

        if run_causal:
            alpha=fixed_relative_scale(rm,ro,hist,CORE.causal_relative_m1_displacement)
            row.update(finite_history_locality(ctx,hist,alpha))
            baseline,_=finite_future_branch(task,dt,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[cp],sch,None,alpha,0)
            branches={}
            for name,axis in [("history",hist),("parallel",hpar),("perp",hperp)]:
                for sign in (-1,+1):
                    print("CAUSAL rep",rep,name,"sign",sign,flush=True)
                    branches[(name,sign)],_=finite_future_branch(task,dt,"resnet",PRIMARY_MODEL_SCALE,rep,ts,bank[cp],sch,axis,alpha,sign)
            h=CORE.primary_future_horizon
            def sec(name): return (branches[(name,+1)][h]-branches[(name,-1)][h])/2.0
            paired=causal_effect_metrics(sec("history"),sec("parallel"),sec("perp"))
            row.update({f"future_{k}":v for k,v in paired.items()})
            for sign,label in ((+1,"plus"),(-1,"minus")):
                signed=causal_effect_metrics(*[branches[(name,sign)][h]-baseline[h] for name in ("history","parallel","perp")])
                row.update({f"future_{label}_{k}":v for k,v in signed.items()})
            even=(branches[("history",+1)][h].double()+branches[("history",-1)][h].double())/2-baseline[h].double()
            row["future_even_rmse"]=float(torch.linalg.vector_norm(even))/math.sqrt(max(1,even.numel()))
            unsigned_mean_norm=(row["future_plus_full_norm"]+row["future_minus_full_norm"])/2
            row["future_even_fraction"]=float(torch.linalg.vector_norm(even))/max(unsigned_mean_norm,1e-30)
            row["future_effect_nontrivial_pass"]=bool(all(row[f"future_{prefix}full_rmse"]>=CORE.causal_min_future_rmse for prefix in ("","plus_","minus_")))
            row["causal_alpha"]=float(alpha)
            row["causal_relative_m1_displacement_full"]=float(alpha*_axis_norm(hist)/max(optimizer_moment_norms(ro)[0],1e-30))
            row.update(save_causal_endpoints(rep,selected_k,alpha,baseline,branches))

    del rm,ro,dm,do,ctx,ctx_transfer,controls,geom,transfer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return row

def discovery_select_rank(df):
    validate_task_frame(df,DISCOVERY_IDS,"discovery rank selection")
    records=[]
    chosen=None
    for k in CORE.rank_candidates:
        vals=df[f"capture_transfer_k{k}"].to_numpy(float)
        rec=mean_ci_record(vals,stable_seed("S7_rank_ci",k))
        rec["k"]=int(k); records.append(rec)
        if chosen is None and rec["mean"]>=CORE.rank_discovery_mean_capture and rec["ci95"][0]>=CORE.rank_discovery_ci_lower:
            chosen=int(k)
    prequalified=chosen is not None
    if chosen is None: chosen=int(max(CORE.rank_candidates))
    return chosen,prequalified,records

def competency_summary(df,include_locality=False):
    final=df["final_acc"].to_numpy(float); recip=df["recipient_acc"].to_numpy(float)
    final_frac=float(np.mean(final>=CORE.clean_task_nontrivial_floor)); recip_frac=float(np.mean(recip>=CORE.recipient_task_nontrivial_floor))
    passed=bool(np.median(final)>=CORE.clean_median_acc_floor and np.median(recip)>=CORE.recipient_median_acc_floor and
                final_frac>=CORE.min_nontrivial_task_fraction and recip_frac>=CORE.min_nontrivial_task_fraction)
    out={"passed":passed,"median_final_acc":float(np.median(final)),"median_recipient_acc":float(np.median(recip)),
         "final_nontrivial_fraction":final_frac,"recipient_nontrivial_fraction":recip_frac}
    if include_locality:
        values=df["finite_locality_pass"]
        if values.isna().any() or not values.map(lambda v:isinstance(v,(bool,np.bool_))).all():
            raise RuntimeError("Invalid finite-locality flags; refusing truthy strings/NaN.")
        loc=values.to_numpy(dtype=bool); frac=float(np.mean(loc))
        effect=float(np.mean(df["full_one_step_rmse"].to_numpy(float)>=CORE.causal_min_one_step_rmse))
        out.update({"locality_pass_fraction":frac,"nontrivial_effect_fraction":effect})
        future=df["future_effect_nontrivial_pass"]
        if future.isna().any() or not future.map(lambda v:isinstance(v,(bool,np.bool_))).all(): raise RuntimeError("Invalid future-effect qualification flags.")
        measured=np.vstack([df[f"future_{prefix}full_rmse"].to_numpy(float) for prefix in ("","plus_","minus_")])
        measured_pass=np.isfinite(measured).all(axis=0)&(measured>=CORE.causal_min_future_rmse).all(axis=0)
        if not np.array_equal(measured_pass,future.to_numpy(dtype=bool)): raise RuntimeError("Future-effect flags disagree with measured effects.")
        future_frac=float(np.mean(measured_pass)); out["future_nontrivial_effect_fraction"]=future_frac
        out["passed"]=bool(out["passed"] and frac>=CORE.hidden_locality_required_fraction and effect>=CORE.hidden_locality_required_fraction and future_frac>=CORE.hidden_locality_required_fraction)
    return out

def confirmation_claims(df,selected_k):
    validate_task_frame(df,CONFIRMATION_IDS,"confirmation claims")
    if selected_k not in CORE.rank_candidates: raise RuntimeError("Confirmation rank is not a frozen candidate.")
    foundation=competency_summary(df,include_locality=True); H7F=bool(foundation["passed"])
    cap=df[f"capture_transfer_k{selected_k}"].to_numpy(float)
    caprec=mean_ci_record(cap,stable_seed("H7L_capture"))
    task_frac=float(np.mean(cap>=CORE.confirmation_capture_per_task_min))
    H7L=bool(selected_k<=CORE.low_rank_max_k and caprec["mean"]>=CORE.confirmation_capture_mean_min and
             caprec["ci95"][0]>=CORE.confirmation_capture_ci_lower_min and task_frac>=CORE.confirmation_capture_task_fraction_min)

    align=df["history_alignment_selected"].to_numpy(float)
    excess=df["alignment_excess_vs_random"].to_numpy(float)
    pot=df["history_random_potency_ratio"].to_numpy(float)
    best=df["history_best_random_potency_ratio"].to_numpy(float)
    shuf=df["history_signshuffle_potency_ratio"].to_numpy(float)
    perm=df["history_permute_potency_ratio"].to_numpy(float)
    alignrec=mean_ci_record(align,stable_seed("H7A_align")); exrec=mean_ci_record(excess,stable_seed("H7A_excess"))
    potrec=mean_ci_record(pot,stable_seed("H7A_pot")); bestrec=mean_ci_record(best,stable_seed("H7A_best")); shrec=mean_ci_record(shuf,stable_seed("H7A_shuf")); perec=mean_ci_record(perm,stable_seed("H7A_perm"))
    logpot=np.log(np.maximum(pot,1e-30)); logbest=np.log(np.maximum(best,1e-30)); logsh=np.log(np.maximum(shuf,1e-30)); logpe=np.log(np.maximum(perm,1e-30))
    pvals={"alignment_excess":one_sided_signflip_p(excess,stable_seed("H7A_p_excess")),
           "random_log_ratio":one_sided_signflip_p(logpot,stable_seed("H7A_p_rand")),
           "best_random_log_ratio":one_sided_signflip_p(logbest,stable_seed("H7A_p_best")),
           "shuffle_log_ratio":one_sided_signflip_p(logsh,stable_seed("H7A_p_shuf")),
           "permute_log_ratio":one_sided_signflip_p(logpe,stable_seed("H7A_p_perm"))}
    H7A=bool(alignrec["mean"]>=CORE.alignment_history_mean_min and exrec["ci95"][0]>=CORE.alignment_excess_ci_lower_min and
             potrec["mean"]>=CORE.potency_random_ratio_mean_min and potrec["ci95"][0]>=CORE.potency_random_ratio_ci_lower_min and
             bestrec["mean"]>=CORE.potency_best_random_ratio_mean_min and bestrec["ci95"][0]>=CORE.potency_best_random_ratio_ci_lower_min and
             shrec["ci95"][0]>=CORE.shuffled_ratio_ci_lower_min and perec["ci95"][0]>=CORE.permuted_ratio_ci_lower_min and
             all(v<0.05 for v in pvals.values()))

    ret=mean_ci_record(df["future_parallel_norm_ratio"],stable_seed("H7C_ret"))
    cos=mean_ci_record(df["future_parallel_cosine"],stable_seed("H7C_cos"))
    perp=mean_ci_record(df["future_perp_norm_ratio"],stable_seed("H7C_perp"))
    pnorm=mean_ci_record(df["parallel_norm_fraction"],stable_seed("H7C_pnorm"))
    add=mean_ci_record(df["future_additivity_error"],stable_seed("H7C_add"))
    paired_pass=causal_gate_from_records(ret,cos,perp,add)
    signed_records={}
    for label in ("plus","minus"):
        records={name:mean_ci_record(df[f"future_{label}_{column}"],stable_seed("H7C_signed",label,name)) for name,column in
                 (("parallel_retention","parallel_norm_ratio"),("parallel_cosine","parallel_cosine"),("perp_ratio","perp_norm_ratio"),("additivity_error","additivity_error"))}
        records["passed"]=causal_gate_from_records(records["parallel_retention"],records["parallel_cosine"],records["perp_ratio"],records["additivity_error"])
        signed_records[label]=records
    H7C=bool(paired_pass and all(r["passed"] for r in signed_records.values()))
    grad=mean_ci_record(df["history_gradient_potency_ratio"],stable_seed("H7A_grad"))
    pre=mean_ci_record(df["history_preconditioned_gradient_potency_ratio"],stable_seed("H7A_pregrad"))
    return {
        "claims":{"H7_F_foundation":H7F,"H7_L_low_rank_control":H7L,"H7_A_history_alignment_privilege":H7A,"H7_C_causal_sufficiency":H7C,
                  "H7_M_strong_mechanistic_claim":bool(H7F and H7L and H7A and H7C)},
        "H7_F":foundation,
        "H7_L":{"selected_k":int(selected_k),"independent_anchor_capture":caprec,"task_fraction_capture_ge_min":task_frac},
        "H7_A":{"history_alignment":alignrec,"alignment_excess_vs_random":exrec,"history_random_potency_ratio":potrec,"history_best_random_potency_ratio":bestrec,
                "history_signshuffle_potency_ratio":shrec,"history_permute_potency_ratio":perec,"one_sided_signflip_p":pvals,
                "history_gradient_potency_ratio_diagnostic":grad,"history_preconditioned_gradient_potency_ratio_diagnostic":pre},
        "H7_C":{"paired_passed":paired_pass,"parallel_retention":ret,"parallel_cosine":cos,"perp_ratio":perp,"parallel_norm_fraction_diagnostic":pnorm,"additivity_error":add,"signed_baseline_relative":signed_records,"even_fraction_diagnostic":mean_ci_record(df["future_even_fraction"],stable_seed("H7C_even"))}
    }


In [ ]:

# 9. Zero-hour integrity tests: optimizer semantics, JVP/VJP adjointness, known-rank discrimination, invariance
def _manual_adamw_equivalence_qa():
    rep=990001
    task=make_stage07_task(rep); ts=s7_train_spec()
    # Small QA checkpoint at first temporal point; research protocol itself is unchanged.
    bank,sch,dt,_=build_checkpoint_bank(task,"resnet",1,rep,ts)
    cp=checkpoint_step_for(CORE.temporal_fracs[0],ts.steps)
    ctx=build_control_operator(task,dt,"resnet",1,rep,ts,bank[cp],sch,cp)
    m2,o2=load_snapshot(task,"resnet",1,rep,ts,bank[cp])
    train_step(m2,o2,dt,sch[cp],cp,"resnet",1,rep)
    actual=dict(m2.named_parameters())
    diff=max(float(torch.max(torch.abs(actual[n].detach()-ctx["manual_next_params"][n])).cpu()) for n in actual)
    if diff>CORE.manual_adamw_param_max_abs_diff: raise RuntimeError(f"Manual AdamW map mismatch: {diff}")
    # Adjoint identity <Jv,u> = <v,J^T u>
    hist=None
    dp=donor_step_for(CORE.temporal_fracs[0],ts.steps)
    dm,do=load_snapshot(task,"resnet",1,rep,ts,bank[dp]); rm,ro=load_snapshot(task,"resnet",1,rep,ts,bank[cp])
    hist=_axis_normalize(_history_axis(rm,ro,dm,do))
    u=_rademacher_like_output(ctx["base_output"],stable_seed("S7_adjoint_qa"),normalize=True)
    Jv=operator_jvp(ctx,hist); JTu=operator_vjp(ctx,u)
    lhs=float(torch.dot(Jv.double(),u.double()).cpu()); rhs=_axis_dot(hist,JTu)
    rel=abs(lhs-rhs)/max(abs(lhs)+abs(rhs),1e-12)
    if rel>CORE.adjoint_rel_error_max: raise RuntimeError(f"JVP/VJP adjoint QA failed: {rel}")
    # Finite difference directional derivative.
    eps=1e-3
    zp=ctx["op"](_axis_scale(hist,+eps)).detach(); zm=ctx["op"](_axis_scale(hist,-eps)).detach()
    fd=(zp-zm)/(2*eps); frel=float(torch.linalg.vector_norm(fd-Jv)/max(torch.linalg.vector_norm(Jv),torch.tensor(1e-12)))
    if frel>CORE.finite_diff_rel_error_max: raise RuntimeError(f"Finite-difference JVP QA failed: {frel}")
    # Function-preserving m1 surgery.
    before=model_hash(rm); z0=_centered_logits(rm,dt["roles"]["response"]["X"])
    apply_m1_axis_surgery(rm,ro,hist,1e-4)
    if model_hash(rm)!=before or float(torch.max(torch.abs(_centered_logits(rm,dt["roles"]["response"]["X"])-z0)))!=0.0:
        raise RuntimeError("Function-preserving surgery QA failed.")
    del task,bank,sch,dt,ctx,m2,o2,dm,do,rm,ro
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {"manual_adamw_max_abs_diff":diff,"adjoint_rel_error":rel,"finite_diff_rel_error":frel}

def _synthetic_rank_qa():
    # Exercise the actual instrument, not an exact SVD in place of its estimator.
    out={}; generator=torch.Generator(device="cpu").manual_seed(1234)
    ranks=(1,4,12) if PROFILE=="research" else (1,4)
    def context(matrix):
        def op(delta): return matrix@delta["axis"]
        return {"op":op,"zero":{"axis":torch.zeros(matrix.shape[1],device=matrix.device)},
                "base_output":torch.zeros(matrix.shape[0],device=matrix.device)}
    for rank in ranks:
        L=torch.linalg.qr(torch.randn(64,rank,generator=generator)).Q
        R=torch.linalg.qr(torch.randn(96,rank,generator=generator)).Q
        matrix=(L@torch.diag(torch.linspace(1.,.8,rank))@R.T).to(DEVICE)
        geom=estimate_local_control_geometry(context(matrix),990100+rank)
        captured=sum(float(operator_jvp(context(matrix),u).double().square().sum().cpu()) for u in geom["basis"][:rank])/float(matrix.double().square().sum().cpu())
        if captured<.999: raise RuntimeError(f"Actual known-rank recovery failed: {rank}, {captured}")
        out[f"exact_rank_{rank}"]={"actual_capture":captured}
    L=torch.linalg.qr(torch.randn(64,64,generator=generator)).Q
    E=torch.linalg.qr(torch.randn(64,64,generator=generator)).Q
    R=torch.linalg.qr(torch.randn(96,64,generator=generator)).Q
    for fraction in (.85,4/64):
        s=torch.cat((torch.full((4,),math.sqrt(fraction/4)),torch.full((60,),math.sqrt((1-fraction)/60))))
        fit=((L*s)@R.T).to(DEVICE); evaluation=((E*s)@R.T).to(DEVICE)
        geom=estimate_local_control_geometry(context(fit),990200+int(fraction*10000))
        captured=sum(float(operator_jvp(context(evaluation),u).double().square().sum().cpu()) for u in geom["basis"][:4])/float(evaluation.double().square().sum().cpu())
        if fraction==.85 and captured<.82: raise RuntimeError(f"Weak-tail instrument calibration failed: {captured}")
        if fraction!=.85 and captured>.15: raise RuntimeError(f"Flat-spectrum discrimination failed: {captured}")
        out[f"top4_true_{fraction}"]={"actual_capture":captured}
    return out

def _causal_baseline_qa():
    def response(x,y): return torch.tensor([float(x),100*float(y)**2])
    directions=((1,1),(1,0),(0,1))
    paired=causal_effect_metrics(*[(response(x,y)-response(-x,-y))/2 for x,y in directions])
    signed=causal_effect_metrics(*[response(x,y)-response(0,0) for x,y in directions])
    def gate(metrics):
        records=[{"mean":metrics[k],"ci95":[metrics[k],metrics[k]]} for k in ("parallel_norm_ratio","parallel_cosine","perp_norm_ratio","additivity_error")]
        return causal_gate_from_records(*records)
    if not gate(paired) or gate(signed): raise RuntimeError("Causal baseline counterexample was not correctly distinguished.")
    return {"paired_gate":gate(paired),"baseline_relative_gate":gate(signed),"passed":True}

def run_zero_hour_integrity():
    qa={"protocol_hash":PROTOCOL_HASH,"optimizer_operator":_manual_adamw_equivalence_qa(),"synthetic_rank":_synthetic_rank_qa(),"causal_baseline":_causal_baseline_qa(),"passed":True}
    atomic_json(BASE/"manifests"/"zero_hour_integrity.json",qa)
    print("STAGE07 ZERO-HOUR INTEGRITY PASSED",json.dumps(qa,indent=2))
    return qa


In [ ]:
# 10. Frozen scientific-code semantic hash shared by discovery and confirmation
SCIENTIFIC_CODE_SCHEMA_VERSION="S7_SCIENTIFIC_ID_V2"
SCIENTIFIC_NAMES=[
    "file_sha256","model_hash","optimizer_hash","optimizer_defaults_record","parameter_count","_gn","hash_tree","tree_to_cpu","_finite_vector","_validated_capture","validate_task_frame","stable_seed","bootstrap_ci","mean_ci_record","one_sided_signflip_p","SuiteSpec","spatial_basis","make_suite","nested_clean_labels","nested_noisy_labels","render_fixedK","render_nuisanceK","make_task_variant",
    "ConvCalib","BasicBlockGN","ResNet18GN","ViT","build_model","TrainSpec","s7_train_spec","resolved_checkpoints","paired_stream_id","make_schedule","make_optimizer","step_rng_seed","set_step_rng","prepare_device_task","train_step","snapshot","load_snapshot","build_checkpoint_bank",
    "forward_bundle","evaluate_public","optimizer_moment_norms","_axis_norm","_axis_dot","_axis_scale","_axis_add","_axis_normalize","_axis_linear_combination","_axis_project","_axis_projection_fraction","_orthonormalize_axes","_history_axis","_match_tensor_norms","build_strong_controls",
    "_optimizer_state_maps","build_control_operator","operator_vjp","operator_jvp","operator_axis_potency","_rademacher_like_output","_random_axis_like","_normalized_control_range","_complete_control_basis","estimate_local_control_geometry","evaluate_subspace_transfer","finite_history_locality",
    "apply_m1_axis_surgery","fixed_relative_scale","_centered_logits","finite_future_branch","make_stage07_task","donor_step_for","checkpoint_step_for","temporal_history_panel","causal_effect_metrics","causal_gate_from_records","save_causal_endpoints","verify_causal_endpoints","main_geometry_measurement","competency_summary","discovery_select_rank","confirmation_claims"
]
def _stable_const(x):
    if isinstance(x,types.CodeType): return {"code":_stable_code_identity(x)}
    if isinstance(x,(str,int,float,bool,type(None),bytes)): return repr(x)
    if isinstance(x,tuple): return ["tuple",[_stable_const(v) for v in x]]
    if isinstance(x,frozenset): return ["frozenset",sorted((_stable_const(v) for v in x),key=lambda z:json.dumps(z,sort_keys=True,default=str))]
    return f"<{type(x).__module__}.{type(x).__qualname__}:{repr(x)}>"

def _stable_code_identity(c):
    return {
        "co_code":c.co_code.hex(),
        "co_consts":[_stable_const(v) for v in c.co_consts],
        "co_names":list(c.co_names),"co_varnames":list(c.co_varnames),
        "co_freevars":list(c.co_freevars),"co_cellvars":list(c.co_cellvars),
        "co_argcount":c.co_argcount,"co_posonlyargcount":getattr(c,"co_posonlyargcount",0),
        "co_kwonlyargcount":c.co_kwonlyargcount,"co_nlocals":c.co_nlocals,"co_flags":c.co_flags,
    }

def _function_code_identity(fn):
    return {"code":_stable_code_identity(fn.__code__),"defaults":_stable_const(fn.__defaults__),
            "kwdefaults":{k:_stable_const(v) for k,v in sorted((fn.__kwdefaults__ or {}).items())}}

def _object_code_identity(obj):
    if isinstance(obj,types.FunctionType):
        return {"kind":"function","identity":_function_code_identity(obj)}
    if isinstance(obj,type):
        methods={}; attrs={}
        for k,v in sorted(vars(obj).items()):
            if isinstance(v,types.FunctionType): methods[k]=_function_code_identity(v)
            elif not k.startswith("__") and isinstance(v,(str,int,float,bool,type(None),tuple)):
                attrs[k]=repr(v)
        fields=getattr(obj,"__dataclass_fields__",None)
        field_defaults={}
        if fields:
            from dataclasses import MISSING
            for k,f in sorted(fields.items()):
                if f.default is MISSING:
                    dv="<MISSING>"
                elif isinstance(f.default,(str,int,float,bool,type(None),tuple)):
                    dv=repr(f.default)
                else:
                    dv=f"<{type(f.default).__module__}.{type(f.default).__qualname__}>"
                field_defaults[k]=dv
        return {"kind":"class","methods":methods,"attrs":attrs,"dataclass_field_defaults":field_defaults,"bases":[b.__module__+"."+b.__qualname__ for b in obj.__bases__]}
    raise TypeError(type(obj))

def _semantic_code_hash():
    payload={n:_object_code_identity(globals()[n]) for n in SCIENTIFIC_NAMES}
    payload["schema_version"]=SCIENTIFIC_CODE_SCHEMA_VERSION
    payload["core"]=asdict(CORE); payload["protocol_hash"]=PROTOCOL_HASH
    payload["scientific_constants"]={"master_seed":MASTER_SEED,"primary_model_scale":PRIMARY_MODEL_SCALE,"role_names":list(ROLE_NAMES)}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()
SCIENTIFIC_CODE_HASH=_semantic_code_hash()
initialize_campaign_manifests()
atomic_json(BASE/"manifests"/"scientific_code_identity.json",{"sha256":SCIENTIFIC_CODE_HASH,"schema_version":SCIENTIFIC_CODE_SCHEMA_VERSION,"names":SCIENTIFIC_NAMES,"protocol_hash":PROTOCOL_HASH,"env_compat_hash":ENV_COMPAT_HASH})
print("SCIENTIFIC_CODE_HASH",SCIENTIFIC_CODE_HASH)


In [ ]:

# 10.5 Research workload contract (does not change scientific workload)
def stage07_workload_contract_discovery():
    n=len(DISCOVERY_IDS)
    rec={
      "phase":"discovery","n_tasks":n,"task_ids":list(map(int,DISCOVERY_IDS)),
      "base_training_updates":int(n*CORE.train_steps),
      "temporal_checkpoints_per_task":len(CORE.temporal_fracs),
      "temporal_history_plus_random_jvps_per_task":int(len(CORE.temporal_fracs)*(1+CORE.tensor_random_controls)),
      "basis_vjps_per_task":int(CORE.basis_probes),
      "basis_ritz_jvps_per_task_max":int(max(CORE.basis_probes,max(CORE.rank_candidates))),
      "power_iterations":CORE.basis_power_iterations,
      "power_jvps_per_task_max":int(CORE.basis_power_iterations*max(CORE.basis_probes,max(CORE.rank_candidates))),
      "power_vjps_per_task_max":int(CORE.basis_power_iterations*max(CORE.basis_probes,max(CORE.rank_candidates))),
      "independent_trace_vjps_per_task":int(CORE.trace_probes),
      "independent_transfer_trace_vjps_per_task":int(CORE.transfer_trace_probes),
      "transfer_basis_jvps_per_task":int(max(CORE.rank_candidates)),
      "main_history_and_control_jvps_per_task_per_anchor":int(1+CORE.tensor_random_controls+4),
      "main_history_and_control_jvps_two_anchors_per_task":int(2*(1+CORE.tensor_random_controls+4)),
      "rank_candidates":list(map(int,CORE.rank_candidates)),
      "research_model":"ResNet18-GN scale 2" if PROFILE=="research" else "ResNet18-GN scale 1 (smoke only)",
      "persistence":PERSISTENCE_ACTIVE,
      "soft_runtime_guard_hours":CORE.runtime_soft_limit_hours,
      "runtime_policy":"Never reduce tasks/probes to fit runtime. If the soft guard is reached, stop only between committed tasks and resume the same notebook."
    }
    atomic_json(BASE/"manifests"/"stage07_discovery_workload_contract.json",rec)
    print("STAGE07 DISCOVERY WORKLOAD",json.dumps(rec,indent=2))
    return rec
WORKLOAD_CONTRACT=stage07_workload_contract_discovery()


In [ ]:

# 11A. Paid Stage07 discovery campaign: 20 fresh tasks, rank selection, frozen discovery lock
DISCOVERY_DIR=BASE/"discovery"

def _rep_paths(rep,cohort="discovery"):
    d=BASE/cohort
    return d/f"rep_{int(rep)}_geometry", d/f"rep_{int(rep)}_temporal"

def _verify_discovery_commit(rep,gp,tp,commit):
    if commit.get("protocol_hash")!=PROTOCOL_HASH or commit.get("scientific_code_hash")!=SCIENTIFIC_CODE_HASH or commit.get("env_compat_hash")!=ENV_COMPAT_HASH:
        raise RuntimeError(f"Discovery commit provenance mismatch for rep {rep}")
    gpath,tpath=resolve_table_path(gp),resolve_table_path(tp)
    if file_sha256(gpath)!=commit.get("geometry_sha256") or file_sha256(tpath)!=commit.get("temporal_sha256"):
        raise RuntimeError(f"Discovery committed artifact hash mismatch for rep {rep}")
    verify_task_tables(rep,gp,tp,commit,"discovery")

def _load_or_run_discovery_rep(rep):
    gp,tp=_rep_paths(rep,"discovery")
    done=DISCOVERY_DIR/f"rep_{int(rep)}_commit.json"
    if done.exists():
        c=json.loads(done.read_text())
        _verify_discovery_commit(rep,gp,tp,c)
        return load_table(gp).iloc[0].to_dict(), load_table(tp).to_dict("records")
    guard_runtime_before_new_paid_task()
    task=make_stage07_task(rep); ts=s7_train_spec()
    print("START discovery rep",rep,"training",flush=True)
    t0=time.perf_counter(); bank,sch,dt,final_eval=build_checkpoint_bank(task,"resnet",PRIMARY_MODEL_SCALE,rep,ts)
    print("MEASURE discovery rep",rep,"temporal panel",flush=True)
    temporal=temporal_history_panel(task,dt,bank,sch,rep,ts)
    print("MEASURE discovery rep",rep,"control geometry",flush=True)
    row=main_geometry_measurement(task,dt,bank,sch,rep,ts,selected_k=None,run_causal=False)
    row.update({"final_acc":float(final_eval["acc"]),"protocol_hash":PROTOCOL_HASH,"scientific_code_hash":SCIENTIFIC_CODE_HASH,"env_compat_hash":ENV_COMPAT_HASH,"seconds":time.perf_counter()-t0})
    validate_task_frame(pd.DataFrame([row]),(int(rep),),"A task before commit")
    validate_temporal_frame(pd.DataFrame(temporal),rep,"A task before commit")
    p1=save_table(pd.DataFrame([row]),gp); p2=save_table(pd.DataFrame(temporal),tp)
    commit={"replicate_id":int(rep),"task_hash":str(row["task_hash"]),"protocol_hash":PROTOCOL_HASH,"scientific_code_hash":SCIENTIFIC_CODE_HASH,"env_compat_hash":ENV_COMPAT_HASH,
            "geometry_sha256":file_sha256(p1),"temporal_sha256":file_sha256(p2),"seconds":float(row["seconds"])}
    atomic_json(done,commit)
    print("DURABLE discovery rep",rep,"sec",round(row["seconds"],1))
    return row,temporal

def run_stage07_discovery():
    run_zero_hour_integrity()
    rows=[]; temp=[]
    for rep in DISCOVERY_IDS:
        r,t=_load_or_run_discovery_rep(rep); rows.append(r); temp.extend(t)
    df=pd.DataFrame(rows); tdf=pd.DataFrame(temp)
    if df["task_hash"].nunique()!=len(df):
        raise RuntimeError("Discovery cohort contains duplicate task hashes; independence violated.")
    validate_task_frame(df,DISCOVERY_IDS,"discovery campaign")
    discovery_task_hashes=sorted(map(str,df["task_hash"].tolist()))
    foundation=competency_summary(df,include_locality=False)
    selected_k,prequalified,curves=discovery_select_rank(df)
    summary={"protocol_hash":PROTOCOL_HASH,"scientific_code_hash":SCIENTIFIC_CODE_HASH,"env_compat_hash":ENV_COMPAT_HASH,"n_discovery":len(df),
             "foundation":foundation,"selected_k":int(selected_k),"low_rank_prequalified":bool(prequalified),"rank_selection_curve":curves,
             "temporal_history_random_ratio":{str(f):mean_ci_record(tdf.loc[np.isclose(tdf.time_frac,float(f)),"history_random_ratio"],
                                                                    stable_seed("S7_temp",f)) for f in CORE.temporal_fracs}}
    lock_payload={"lock_type":"NEXUS-III Stage07 discovery lock","protocol_hash":PROTOCOL_HASH,
                  "scientific_code_hash":SCIENTIFIC_CODE_HASH,"env_compat_hash":ENV_COMPAT_HASH,"selected_k":int(selected_k),
                  "discovery_foundation":foundation,"low_rank_prequalified":bool(prequalified),"rank_selection_curve":curves,
                  "discovery_ids":list(map(int,DISCOVERY_IDS)),"confirmation_ids":list(map(int,CONFIRMATION_IDS)),
                  "discovery_task_hashes":discovery_task_hashes,
                  "selection_rule":PREREG["rank_selection"]}
    lock_payload["payload_sha256"]=hashlib.sha256(json.dumps(lock_payload,sort_keys=True,separators=(",",":")).encode()).hexdigest()
    lock_path=BASE/"locks"/"stage07_discovery_lock.json"
    if lock_path.exists():
        old=json.loads(lock_path.read_text())
        if old!=lock_payload: raise RuntimeError("Existing discovery lock differs from recomputed frozen lock; refusing overwrite.")
    else:
        atomic_json(lock_path,lock_payload)
    save_table(df,BASE/"discovery"/"stage07_discovery_summary")
    save_table(tdf,BASE/"discovery"/"stage07_discovery_temporal")
    atomic_json(BASE/"discovery"/"stage07_discovery_analysis.json",summary)
    export_stage07("DISCOVERY")
    print("DISCOVERY LOCKED",json.dumps(lock_payload,indent=2))
    return summary

def export_stage07(label):
    payload_files=[]
    for p in sorted(BASE.rglob("*")):
        if not p.is_file(): continue
        rel=p.relative_to(BASE)
        if rel.parts and rel.parts[0]=="exports": continue
        if rel.as_posix()=="manifests/artifact_manifest.json": continue
        payload_files.append((p,rel))
    manifest={"protocol_hash":PROTOCOL_HASH,"scientific_code_hash":SCIENTIFIC_CODE_HASH,"env_compat_hash":ENV_COMPAT_HASH,
              "files":[{"path":rel.as_posix(),"bytes":p.stat().st_size,"sha256":file_sha256(p)} for p,rel in payload_files]}
    atomic_json(BASE/"manifests"/"artifact_manifest.json",manifest)
    payload_files.append((BASE/"manifests"/"artifact_manifest.json",Path("manifests/artifact_manifest.json")))
    out=BASE/"exports"/f"NEXUSIII_v2.0.2_STAGE07_{label}_{PROTOCOL_HASH[:10]}_{SCIENTIFIC_CODE_HASH[:10]}.zip"
    tmp=Path(str(out)+".tmp")
    with zipfile.ZipFile(tmp,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p,rel in sorted(payload_files,key=lambda x:x[1].as_posix()):
            info=zipfile.ZipInfo(rel.as_posix(),date_time=(2020,1,1,0,0,0)); info.compress_type=zipfile.ZIP_DEFLATED
            info.external_attr=0o644<<16
            z.writestr(info,p.read_bytes(),compress_type=zipfile.ZIP_DEFLATED,compresslevel=6)
    os.replace(tmp,out)
    with zipfile.ZipFile(out) as z:
        bad=z.testzip()
        if bad is not None: raise RuntimeError(f"ZIP CRC failure: {bad}")
    receipt={"file":out.name,"sha256":file_sha256(out),"bytes":out.stat().st_size,"protocol_hash":PROTOCOL_HASH,
             "scientific_code_hash":SCIENTIFIC_CODE_HASH,"env_compat_hash":ENV_COMPAT_HASH,"label":label}
    atomic_json(Path(str(out)+".receipt.json"),receipt)
    print("FINAL ZIP",out,"sha256",receipt["sha256"])
    return out

RUN_STAGE07=bool(int(os.environ.get("NEXUS_RUN_STAGE07","1")))
if RUN_STAGE07:
    DISCOVERY_RESULT=run_stage07_discovery()
else:
    print("NEXUS_RUN_STAGE07=0: definitions + integrity code loaded; paid discovery not launched.")
